# min

In [21]:

import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
from datetime import datetime, timedelta
from pathlib import Path
from scipy.interpolate import griddata
from tqdm import tqdm
import requests
from io import BytesIO
import pytz
from msal import ConfidentialClientApplication


def round_to_half_degree(value):
    return round(value * 2) / 2


def round_to_quarter_degree(value):
    return round(value * 8) / 8


def find_nearest_index(array, value):
    return (abs(array - value)).argmin()


def get_forecast_locations():
    if "get_city_locations" in globals():
        return get_city_locations()

    if "locations" in globals() and isinstance(locations, dict) and locations:
        return locations

    return {
        'Augsburg': {'latitude': 48.5, 'longitude': 11.0},
        'Karlsruhe': {'latitude': 49.0, 'longitude': 8.5},
        'Mannheim': {'latitude': 49.5, 'longitude': 8.5},
        'Bergerac': {'latitude': 45.0, 'longitude': 0.5},
        'Le Mans': {'latitude': 48.0, 'longitude': 0.0},
        'Delhi': {'latitude': 29.0, 'longitude': 77.0},
        'Jamnagar': {'latitude': 22.5, 'longitude': 70.0},
        'Haldia': {'latitude': 22.0, 'longitude': 88.0},
        'Casablanca': {'latitude': 33.5, 'longitude': -7.5},
        'Curitiba': {'latitude': -25.5, 'longitude': -49.0},
        'Suape': {'latitude': -8.5, 'longitude': -35.0},
        'Santos': {'latitude': -24.0, 'longitude': -46.0},
        'Panama': {'latitude': 9.0, 'longitude': -79.5},
        'Niigata': {'latitude': 37.9161, 'longitude': 139.0364},
        'Sendai': {'latitude': 38.2682, 'longitude': 140.8694},
        'Yamaguchi': {'latitude': 34.1859, 'longitude': 131.4714},
        'Kobe': {'latitude': 34.6901, 'longitude': 135.1955},
        'Oita': {'latitude': 33.2396, 'longitude': 131.6092},
        'Aomori': {'latitude': 40.8246, 'longitude': 140.7406},
        'Seoul': {'latitude': 37.5665, 'longitude': 126.9780},
        'Busan': {'latitude': 35.1796, 'longitude': 129.0756},
        'Yeosu': {'latitude': 34.7606, 'longitude': 127.6622},
        'Harbin': {'latitude': 45.75, 'longitude': 126.65},
        'Changchun': {'latitude': 43.88, 'longitude': 125.3228},
        'Shenyang': {'latitude': 41.8057, 'longitude': 123.4315},
        'Hohhot': {'latitude': 40.8422, 'longitude': 111.75},
        'Urumqi': {'latitude': 43.8256, 'longitude': 87.6168},
        'Xining': {'latitude': 36.6167, 'longitude': 101.7667},
        'Lanzhou': {'latitude': 36.0564, 'longitude': 103.8343},
        'Beijing': {'latitude': 39.9042, 'longitude': 116.4074},
        'Tianjin': {'latitude': 39.3434, 'longitude': 117.3616},
        'Shijiazhuang': {'latitude': 38.0428, 'longitude': 114.5149},
        'Taiyuan': {'latitude': 37.8706, 'longitude': 112.5489},
        'Yinchuan': {'latitude': 38.4872, 'longitude': 106.2309},
        'Jinan': {'latitude': 36.6512, 'longitude': 116.9868},
        'Zhengzhou': {'latitude': 34.7466, 'longitude': 113.6253},
        'Xian': {'latitude': 34.3416, 'longitude': 108.9398},
        'Nanjing': {'latitude': 32.0603, 'longitude': 118.7969},
        'Hangzhou': {'latitude': 30.2741, 'longitude': 120.1551},
        'Hefei': {'latitude': 31.8206, 'longitude': 117.2272},
        'Fuzhou': {'latitude': 26.0745, 'longitude': 119.2965},
        'Nanchang': {'latitude': 28.682, 'longitude': 115.8579},
        'Wuhan': {'latitude': 30.5928, 'longitude': 114.3055},
        'Changsha': {'latitude': 28.228, 'longitude': 112.9388},
        'Guangzhou': {'latitude': 23.1291, 'longitude': 113.2644},
        'Nanning': {'latitude': 22.817, 'longitude': 108.3665},
        'Haikou': {'latitude': 20.044, 'longitude': 110.1999},
        'Chongqing': {'latitude': 29.4316, 'longitude': 106.9123},
        'Guiyang': {'latitude': 26.647, 'longitude': 106.6302},
        'Kunming': {'latitude': 25.0389, 'longitude': 102.7183},
        'Chengdu': {'latitude': 30.5728, 'longitude': 104.0668},
        'Shanghai': {'latitude': 31.2304, 'longitude': 121.4737},
        'Kansas-City': {'latitude': 39.0997, 'longitude': -94.5786},
        'Oklahoma-City': {'latitude': 35.4676, 'longitude': -97.5164},
        'Columbia': {'latitude': 34.0007, 'longitude': -81.0348},
        'Tallahassee': {'latitude': 30.4383, 'longitude': -84.2807},
        'Raleigh': {'latitude': 35.7796, 'longitude': -78.6382},
    }


# ------------------------------------------------------------
# DSL API helper – single authenticated request
# ------------------------------------------------------------
def get_dsl_headers():
    tenant_id = os.getenv("DSL_TENANT_ID", "8619c67c-945a-48ae-8e77-35b1b71c9b98")
    client_id = os.getenv("DSL_CLIENT_ID", "5a838d41-4163-42ef-8479-48fe7db6d547")
    scope = os.getenv("DSL_SCOPE", "api://e504750d-ac05-4807-a2e7-8fcd77b247aa/.default")
    secret_env_name = os.getenv("DSL_SECRET_ENV_NAME", "MDS_SECRET")
    client_secret = os.getenv(secret_env_name)

    if not client_secret:
        raise RuntimeError(
            f"Missing environment variable '{secret_env_name}'. Set it before calling the DSL API."
        )

    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = ConfidentialClientApplication(
        client_id=client_id,
        authority=authority,
        client_credential=client_secret,
    )
    token_result = app.acquire_token_for_client(scopes=[scope])
    access_token = token_result.get("access_token")
    if not access_token:
        error_message = token_result.get("error_description") or token_result.get("error") or "Unknown error"
        raise RuntimeError(f"Unable to fetch DSL OAuth token: {error_message}")

    return {
        "Authorization": f"Bearer {access_token}",
        "User-Agent": os.environ.get("USER") or os.environ.get("USERNAME") or "axpo-user",
        "Content-Type": "application/json",
        "Accept-Encoding": "gzip, deflate",
    }


def get_forecast_json_grid(start_date, end_date, parameter, grid_location, model, ens_select=None, timeout=9000):
    """Fetch forecast for an entire grid in ONE API call."""
    headers = get_dsl_headers()

    source = f"{model}&ensSelect={ens_select}" if ens_select else model
    payload = {
        "parameters": parameter,
        "location": grid_location,
        "source": source,
        "datetime": f"{pd.Timestamp(start_date).strftime('%Y-%m-%d')}T00:00:00Z--{pd.Timestamp(end_date).strftime('%Y-%m-%d')}T00:00:00Z:P1D",
        "outputFormat": "json",
    }

    response = requests.post(
        "https://fs_gateway.prod.axponet.ch/dsl/v3/meteo",
        json=payload,
        headers=headers,
        verify=False,
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()


def extract_city_from_grid(grid_json, city, coords, resolution=0.5):
    """Extract the nearest grid point for a city from a grid JSON response."""
    lat = coords['latitude']
    lon = coords['longitude']

    # Find nearest grid point
    best_coord = None
    best_dist = float('inf')

    for coord in grid_json['data'][0]['coordinates']:
        clat = coord['lat']
        clon = coord['lon']
        dist = (clat - lat)**2 + (clon - lon)**2
        if dist < best_dist:
            best_dist = dist
            best_coord = coord

    if best_coord is None:
        print(f"    {city}: no coordinates in grid response")
        return None

    # Check distance isn't too far (max ~1 degree away)
    if best_dist > 2.0:
        print(f"    {city}: nearest point too far (dist²={best_dist:.2f})")
        return None

    # Check if data has valid values (not all NaN)
    values = [item['value'] for item in best_coord['dates']]
    if all(v is None or (isinstance(v, float) and np.isnan(v)) for v in values):
        print(f"    {city}: nearest point has no valid data")
        return None

    records = []
    for item in best_coord['dates']:
        records.append({'date': pd.to_datetime(item['date']), 'temperature': item['value']})
    df = pd.DataFrame.from_records(records).set_index('date')
    return df


def get_grid_for_locations(locations, resolution=0.5):
    """Compute a bounding-box grid string that covers all city locations."""
    lats = [c['latitude'] for c in locations.values()]
    lons = [c['longitude'] for c in locations.values()]

    # Add margin of 1 degree
    lat_min = np.floor(min(lats) - 1)
    lat_max = np.ceil(max(lats) + 1)
    lon_min = np.floor(min(lons) - 1)
    lon_max = np.ceil(max(lons) + 1)

    # Format: "lat_max,lon_min_lat_min,lon_max:res_lat,res_lon"
    grid_str = f"{lat_max},{lon_min}_{lat_min},{lon_max}:{resolution},{resolution}"
    return grid_str


def group_locations_by_region():
    """Group cities into broad regions to keep grid sizes manageable."""
    locations = get_forecast_locations()

    regions = {
        'east_asia': {},   # lon roughly 70-150
        'europe_africa': {},  # lon roughly -10 to 40
        'americas': {},    # lon roughly -130 to -30
    }

    for city, coords in locations.items():
        lon = coords['longitude']
        if lon >= 60:
            regions['east_asia'][city] = coords
        elif lon >= -15 and lon <= 45:
            regions['europe_africa'][city] = coords
        else:
            regions['americas'][city] = coords

    # Remove empty regions
    return {k: v for k, v in regions.items() if v}


def download_forecasts_grid(locations, parameter='t_min_2m_24h:C', model='ecmwf-vareps',
                            ens_select='mean', forecast_days=44, file_prefix='forecast_min'):
    """Download forecasts using ONE grid call per region instead of one call per city."""
    today = datetime.now(pytz.UTC)
    forecast_end = today + timedelta(days=forecast_days)

    save_dir = Path(r"P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts")
    save_dir.mkdir(parents=True, exist_ok=True)

    forecast_file = save_dir / f"{file_prefix}_{today.strftime('%Y%m%d')}.nc"

    if forecast_file.exists():
        print(f"Forecast file {forecast_file} already exists.")
        return

    # Group cities by region to limit grid size
    region_groups = group_locations_by_region()

    city_forecast_temps = {}

    for region_name, region_locs in region_groups.items():
        # Filter to only cities in our locations dict
        region_cities = {c: loc for c, loc in region_locs.items() if c in locations}
        if not region_cities:
            continue

        print(f"\nFetching grid for region '{region_name}' ({len(region_cities)} cities) ...")
        grid_str = get_grid_for_locations(region_cities, resolution=0.5)
        print(f"  Grid: {grid_str}")

        try:
            grid_json = get_forecast_json_grid(
                start_date=today,
                end_date=forecast_end,
                parameter=parameter,
                grid_location=grid_str,
                model=model,
                ens_select=ens_select,
            )
            print(f"  Grid response received ({len(grid_json['data'][0]['coordinates'])} points)")
        except Exception as e:
            print(f"  ERROR fetching grid for {region_name}: {e}")
            continue

        # Extract each city from the grid
        for city, coords in region_cities.items():
            df = extract_city_from_grid(grid_json, city, coords)
            if df is not None:
                df.index.name = 'time'
                city_forecast_temps[city] = df

    if not city_forecast_temps:
        print("No forecast data retrieved.")
        return

    # Build xarray Dataset
    ds = xr.Dataset()
    for city, data in city_forecast_temps.items():
        ds[city] = xr.DataArray(data=data['temperature'].values, coords={'time': data.index}, dims=['time'])

    # Remove timezone
    if 'time' in ds.coords:
        ds['time'] = pd.to_datetime(ds['time'].values).tz_localize(None)
        print('Removed timezone from time coordinate')

    ds.to_netcdf(forecast_file)
    print(f"Forecast data saved to {forecast_file}")


# Run the download – only a few grid API calls instead of 60+ per-city calls
locations = get_forecast_locations()

# Long-range forecast (44 days, ecmwf-vareps) – t_min
download_forecasts_grid(locations, parameter='t_min_2m_24h:C', model='ecmwf-vareps',
                        ens_select='mean', forecast_days=44, file_prefix='forecast_min')

# Short-range forecast (14 days, ecmwf-ens) – t_min
download_forecasts_grid(locations, parameter='t_min_2m_24h:C', model='ecmwf-ens',
                        ens_select='mean', forecast_days=14, file_prefix='forecast_short_min')


Forecast file P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_min_20260528.nc already exists.
Forecast file P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_short_min_20260528.nc already exists.


# max

In [22]:

# Cell 4: Download t_max forecasts using grid approach (reuses functions from Cell 2)

locations = get_forecast_locations()

# Long-range forecast (44 days, ecmwf-vareps) – t_max
download_forecasts_grid(locations, parameter='t_max_2m_24h:C', model='ecmwf-vareps',
                        ens_select='mean', forecast_days=44, file_prefix='forecast_max')

# Short-range forecast (14 days, ecmwf-ens) – t_max
download_forecasts_grid(locations, parameter='t_max_2m_24h:C', model='ecmwf-ens',
                        ens_select='mean', forecast_days=14, file_prefix='forecast_short_max')


Forecast file P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_max_20260528.nc already exists.
Forecast file P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_short_max_20260528.nc already exists.


# history

In [23]:

def download_this_year_data(locations, api_user=None):
    """Download historical observed t_mean:
    - Use LOCAL ERA5 for ALL cities (bulk of history, no API calls)
    - Fill the gap (ERA5 end → today) with ONE API grid call per region
    """
    current_year = dt.datetime.now().year
    today = dt.datetime.now()
    start_date = dt.datetime(current_year - 1, 4, 15)
    end_date = today

    save_dir = Path(r"P:\QFA\TonyWeather\marta\LPG_REPORT\historical")
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = save_dir / f"historical_{current_year}.nc"

    city_ty_temps = {}

    # ── Step 1: Load ALL cities from local ERA5 (no API calls) ──
    print(f"Loading historical from local ERA5 for ALL cities ...")
    era5_dir = Path(r'P:\QFA\TonyWeather\CLIMATOLOGIES\temperature\ERA5\data_0.5x0.5')

    era5_files = []
    for year in range(start_date.year, end_date.year + 1):
        for pattern in [f'C_T2_{year}_detrended.nc', f'C_T2_{year}.nc']:
            p = era5_dir / pattern
            if p.exists():
                era5_files.append(p)
                break

    if not era5_files:
        print("  WARNING: No local ERA5 files found!")
    else:
        ds_era5 = xr.open_mfdataset(era5_files, combine='by_coords')
        ds_era5 = ds_era5.sel(time=slice(str(start_date.date()), str(end_date.date())))
        era5_end = pd.to_datetime(ds_era5.time.values[-1])
        print(f"  ERA5 time range: {str(ds_era5.time.values[0])[:10]} to {str(era5_end)[:10]}")

        for city, coords in locations.items():
            lat = coords['latitude']
            lon = coords['longitude']
            try:
                ts = ds_era5['t_mean_2m_24h'].sel(latitude=lat, longitude=lon, method='nearest')
                df = ts.to_dataframe().reset_index()[['time', 't_mean_2m_24h']]
                df = df.rename(columns={'time': 'date', 't_mean_2m_24h': 'temperature'}).set_index('date')
                df.index.name = 'time'
                df = df.dropna()
                if not df.empty:
                    city_ty_temps[city] = df
            except Exception as e:
                print(f"    {city}: error extracting from ERA5: {e}")

        ds_era5.close()
        print(f"  Loaded {len(city_ty_temps)}/{len(locations)} cities from ERA5")

    # ── Step 2: Fill gap (ERA5 end → today) with API grid calls per region ──
    #    Split into 5-day chunks and cache each chunk locally to avoid re-downloading
    gap_start = era5_end + pd.Timedelta(days=1) if era5_files else pd.Timestamp(start_date)
    if gap_start < pd.Timestamp(end_date):
        print(f"\nFilling gap ({str(gap_start)[:10]} to {str(end_date)[:10]}) from API in 5-day chunks ...")
        region_groups = group_locations_by_region()
        cache_dir = save_dir / "gap_cache"
        cache_dir.mkdir(parents=True, exist_ok=True)

        for region_name, region_locs in region_groups.items():
            region_cities = {c: loc for c, loc in region_locs.items() if c in locations}
            if not region_cities:
                continue

            print(f"  Region '{region_name}' ({len(region_cities)} cities) ...")
            grid_str = get_grid_for_locations(region_cities, resolution=0.5)

            # Break gap into 5-day windows
            chunk_start = pd.Timestamp(gap_start)
            chunk_size = pd.Timedelta(days=5)

            while chunk_start < pd.Timestamp(end_date):
                chunk_end = min(chunk_start + chunk_size - pd.Timedelta(days=1), pd.Timestamp(end_date))
                cache_file = cache_dir / f"{region_name}_{chunk_start.strftime('%Y%m%d')}_{chunk_end.strftime('%Y%m%d')}.csv"

                if cache_file.exists():
                    # Load from cache
                    df_cached = pd.read_csv(cache_file, index_col=0, parse_dates=True)
                    if df_cached.index.tz is not None:
                        df_cached.index = df_cached.index.tz_localize(None)
                    for city in df_cached.columns:
                        if city in region_cities:
                            df_chunk = df_cached[[city]].dropna().rename(columns={city: 'temperature'})
                            df_chunk.index.name = 'time'
                            if city in city_ty_temps:
                                combined = pd.concat([city_ty_temps[city], df_chunk])
                                combined = combined[~combined.index.duplicated(keep='last')]
                                city_ty_temps[city] = combined.sort_index()
                            else:
                                city_ty_temps[city] = df_chunk
                    chunk_start = chunk_end + pd.Timedelta(days=1)
                    continue

                # Download this chunk from API
                try:
                    gap_json = get_forecast_json_grid(
                        start_date=chunk_start,
                        end_date=chunk_end,
                        parameter='t_mean_2m_24h:C',
                        grid_location=grid_str,
                        model='ecmwf-ens',
                        ens_select='median',
                    )
                    print(f"    {chunk_start.strftime('%Y-%m-%d')} to {chunk_end.strftime('%Y-%m-%d')}: OK")
                except Exception as e:
                    print(f"    {chunk_start.strftime('%Y-%m-%d')} to {chunk_end.strftime('%Y-%m-%d')}: ERROR {e}")
                    chunk_start = chunk_end + pd.Timedelta(days=1)
                    continue

                # Extract cities and build a cache dataframe
                cache_records = {}
                for city, coords in region_cities.items():
                    df_gap = extract_city_from_grid(gap_json, city, coords)
                    if df_gap is not None and not df_gap.empty:
                        if df_gap.index.tz is not None:
                            df_gap.index = df_gap.index.tz_localize(None)
                        df_gap.index.name = 'time'
                        cache_records[city] = df_gap['temperature']
                        if city in city_ty_temps:
                            combined = pd.concat([city_ty_temps[city], df_gap])
                            combined = combined[~combined.index.duplicated(keep='last')]
                            city_ty_temps[city] = combined.sort_index()
                        else:
                            city_ty_temps[city] = df_gap

                # Save chunk to cache
                if cache_records:
                    df_cache = pd.DataFrame(cache_records)
                    df_cache.to_csv(cache_file)

                chunk_start = chunk_end + pd.Timedelta(days=1)

    else:
        print("No gap to fill — ERA5 data is up to date.")

    if not city_ty_temps:
        print("No historical data retrieved.")
        return

    print(f"\nTotal cities with historical data: {len(city_ty_temps)}")

    # Build xarray Dataset
    ds = xr.Dataset()
    for city, df in city_ty_temps.items():
        ds[city] = xr.DataArray(data=df['temperature'].values, coords={'time': df.index}, dims=['time'])

    if 'time' in ds.coords:
        ds['time'] = pd.to_datetime(ds['time'].values).tz_localize(None)
        print("Removed timezone from time coordinate")

    def close_open_datasets_for_path(target_path):
        target_path = os.path.normcase(str(target_path))
        for name, obj in list(globals().items()):
            if not isinstance(obj, xr.Dataset):
                continue
            source = os.path.normcase(str(obj.encoding.get('source', '')))
            if source != target_path:
                continue
            try:
                obj.close()
                print(f"Closed open dataset handle before overwrite: {name}")
            except Exception as exc:
                print(f"Could not close dataset handle {name}: {exc}")

    close_open_datasets_for_path(filename)
    ds.to_netcdf(filename, mode='w')
    print(f"Historical data saved to {filename}")


locations = get_forecast_locations()

download_this_year_data(locations)


Loading historical from local ERA5 for ALL cities ...
  ERA5 time range: 2025-04-15 to 2026-04-14
  Loaded 57/57 cities from ERA5

Filling gap (2026-04-15 to 2026-05-28) from API in 5-day chunks ...
  Region 'east_asia' (42 cities) ...
  Region 'europe_africa' (6 cities) ...
  Region 'americas' (9 cities) ...

Total cities with historical data: 57
Removed timezone from time coordinate
Historical data saved to P:\QFA\TonyWeather\marta\LPG_REPORT\historical\historical_2026.nc


In [24]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.dates as mdates
import os
import pytz

def get_city_country_mapping():
    """
    Returns a dictionary mapping cities to their countries and population.
    Ensures all entries have 'country' and 'population' keys.
    """
    mapping = {
        'Augsburg': {'country': 'Germany', 'population': 300000},
        'Karlsruhe': {'country': 'Germany', 'population': 310000},
        'Mannheim': {'country': 'Germany', 'population': 320000},
        'Bergerac': {'country': 'France', 'population': 27000},
        'Le Mans': {'country': 'France', 'population': 143000},
        'Delhi': {'country': 'India', 'population': 19000000},  # Fixed country (was 'Delhi')
        'Jamnagar': {'country': 'India', 'population': 600000},  # Fixed country (was 'Jamnagar')
        'Haldia': {'country': 'India', 'population': 200000},  # Fixed country (was 'Haldia')
        'Casablanca': {'country': 'Morocco', 'population': 3500000},
        'Curitiba': {'country': 'Brazil (Curitiba, Santos)', 'population': 1900000},
        'Suape': {'country': 'Brazil (Suape)', 'population': 30000},
        'Santos': {'country': 'Brazil (Curitiba, Santos)', 'population': 430000},
        'Panama': {'country': 'Panama', 'population': 880000},
        'Niigata': {'country': 'Japan', 'population': 800000},
        'Sendai': {'country': 'Japan', 'population': 1000000},
        'Yamaguchi': {'country': 'Japan', 'population': 145000},
        'Kobe': {'country': 'Japan', 'population': 1500000},
        'Oita': {'country': 'Japan', 'population': 470000},
        'Aomori': {'country': 'Japan', 'population': 280000},
        'Seoul': {'country': 'South Korea', 'population': 9700000},
        'Busan': {'country': 'South Korea', 'population': 3400000},
        'Yeosu': {'country': 'South Korea', 'population': 300000},
        'Harbin': {'country': 'China (North)', 'population': 30290000},
        'Changchun': {'country': 'China (North)', 'population': 23170000},
        'Shenyang': {'country': 'China (North)', 'population': 41550000},
        'Hohhot': {'country': 'China (North)', 'population': 23800000},
        'Urumqi': {'country': 'China (North)', 'population': 26230000},
        'Xining': {'country': 'China (North)', 'population': 5930000},
        'Lanzhou': {'country': 'China (North)', 'population': 24580000},
        'Beijing': {'country': 'China (North)', 'population': 21830000},
        'Tianjin': {'country': 'China (North)', 'population': 13640000},
        'Shijiazhuang': {'country': 'China (North)', 'population': 78780000},
        'Taiyuan': {'country': 'China (North)', 'population': 34460000},
        'Yinchuan': {'country': 'China (North)', 'population': 7290000},
        'Jinan': {'country': 'China (North)', 'population': 100800000},
        'Zhengzhou': {'country': 'China (Central)', 'population': 97850000},
        'Xian': {'country': 'China (North)', 'population': 39520000},
        'Nanjing': {'country': 'China (South)', 'population': 85260000},
        'Hangzhou': {'country': 'China (South)', 'population': 66269999},
        'Hefei': {'country': 'China (Central)', 'population': 61270000},
        'Fuzhou': {'country': 'China (South)', 'population': 41880000},
        'Nanchang': {'country': 'China (Central)', 'population': 45280000},
        'Wuhan': {'country': 'China (Central)', 'population': 58440000},
        'Changsha': {'country': 'China (Central)', 'population': 66040000},
        'Guangzhou': {'country': 'China (South)', 'population': 127060000},
        'Nanning': {'country': 'China (South)', 'population': 50470000},
        'Haikou': {'country': 'China (South)', 'population': 10270000},
        'Chongqing': {'country': 'China (South)', 'population': 32130000},
        'Guiyang': {'country': 'China (South)', 'population': 38560000},
        'Kunming': {'country': 'China (South)', 'population': 46930000},
        'Chengdu': {'country': 'China (South)', 'population': 83470000},
        'Shanghai': {'country': 'China (South)', 'population': 24800000},
        'Kansas-City': {'country': 'USA (Kansas, Oklahoma)', 'population': 510000},
        'Oklahoma-City': {'country': 'USA (Kansas, Oklahoma)', 'population': 650000},
        'Columbia': {'country': 'USA (Columbia)', 'population': 133000},
        'Tallahassee': {'country': 'USA (Tallahassee)', 'population': 194000},
        'Raleigh': {'country': 'USA (Raleigh)', 'population': 480000},
    }
    
    # The original mapping had these entries with lat/lon instead of country/population
    # Adding them correctly to the mapping dictionary
    
    # If there are any cities with missing keys, ensure they have defaults
    for city in mapping:
        if 'country' not in mapping[city]:
            # Use city name as country if missing
            mapping[city]['country'] = 'Unknown'
        if 'population' not in mapping[city]:
            # Use a default population if missing
            mapping[city]['population'] = 100000
            
    return mapping

def get_city_locations():
    return {
        'Augsburg': {'latitude': 48.5, 'longitude': 11.0},
        'Karlsruhe': {'latitude': 49.0, 'longitude': 8.5},
        'Mannheim': {'latitude': 49.5, 'longitude': 8.5},
        'Bergerac': {'latitude': 45.0, 'longitude': 0.5},
        'Le Mans': {'latitude': 48.0, 'longitude': 0.0},
        'Delhi': {'latitude': 29.0, 'longitude': 77.0},
        'Jamnagar': {'latitude': 22.5, 'longitude': 70.0},
        'Haldia': {'latitude': 22.0, 'longitude': 88.0},
        'Casablanca': {'latitude': 33.5, 'longitude': -7.5},
        'Curitiba': {'latitude': -25.5, 'longitude': -49.0},
        'Suape': {'latitude': -8.5, 'longitude': -35.0},
        'Santos': {'latitude': -24.0, 'longitude': -46.0},
        'Panama': {'latitude': 9.0, 'longitude': -79.5},
        'Niigata': {'latitude': 37.9161, 'longitude': 139.0364},
        'Sendai': {'latitude': 38.2682, 'longitude': 140.8694},
        'Yamaguchi': {'latitude': 34.1859, 'longitude': 131.4714},
        'Kobe': {'latitude': 34.6901, 'longitude': 135.1955},
        'Oita': {'latitude': 33.2396, 'longitude': 131.6092},
        'Aomori': {'latitude': 40.8246, 'longitude': 140.7406},
        'Seoul': {'latitude': 37.5665, 'longitude': 126.9780},
        'Busan': {'latitude': 35.1796, 'longitude': 129.0756},
        'Yeosu': {'latitude': 34.7606, 'longitude': 127.6622},
        'Harbin': {'latitude': 45.75, 'longitude': 126.65},
        'Changchun': {'latitude': 43.88, 'longitude': 125.3228},
        'Shenyang': {'latitude': 41.8057, 'longitude': 123.4315},
        'Hohhot': {'latitude': 40.8422, 'longitude': 111.75},
        'Urumqi': {'latitude': 43.8256, 'longitude': 87.6168},
        'Xining': {'latitude': 36.6167, 'longitude': 101.7667},
        'Lanzhou': {'latitude': 36.0564, 'longitude': 103.8343},
        'Beijing': {'latitude': 39.9042, 'longitude': 116.4074},
        'Tianjin': {'latitude': 39.3434, 'longitude': 117.3616},
        'Shijiazhuang': {'latitude': 38.0428, 'longitude': 114.5149},
        'Taiyuan': {'latitude': 37.8706, 'longitude': 112.5489},
        'Yinchuan': {'latitude': 38.4872, 'longitude': 106.2309},
        'Jinan': {'latitude': 36.6512, 'longitude': 116.9868},
        'Zhengzhou': {'latitude': 34.7466, 'longitude': 113.6253},
        'Xian': {'latitude': 34.3416, 'longitude': 108.9398},
        'Nanjing': {'latitude': 32.0603, 'longitude': 118.7969},
        'Hangzhou': {'latitude': 30.2741, 'longitude': 120.1551},
        'Hefei': {'latitude': 31.8206, 'longitude': 117.2272},
        'Fuzhou': {'latitude': 26.0745, 'longitude': 119.2965},
        'Nanchang': {'latitude': 28.682, 'longitude': 115.8579},
        'Wuhan': {'latitude': 30.5928, 'longitude': 114.3055},
        'Changsha': {'latitude': 28.228, 'longitude': 112.9388},
        'Guangzhou': {'latitude': 23.1291, 'longitude': 113.2644},
        'Nanning': {'latitude': 22.817, 'longitude': 108.3665},
        'Haikou': {'latitude': 20.044, 'longitude': 110.1999},
        'Chongqing': {'latitude': 29.4316, 'longitude': 106.9123},
        'Guiyang': {'latitude': 26.647, 'longitude': 106.6302},
        'Kunming': {'latitude': 25.0389, 'longitude': 102.7183},
        'Chengdu': {'latitude': 30.5728, 'longitude': 104.0668},
        'Shanghai': {'latitude': 31.2304, 'longitude': 121.4737},
        'Kansas-City': {'latitude': 39.0997, 'longitude': -94.5786},
        'Oklahoma-City': {'latitude': 35.4676, 'longitude': -97.5164},
        'Columbia': {'latitude': 34.0007, 'longitude': -81.0348},
        'Tallahassee': {'latitude': 30.4383, 'longitude': -84.2807},
        'Raleigh': {'latitude': 35.7796, 'longitude': -78.6382},
    }

def load_and_process_data(historical_file, forecast_files):
    """Load historical and forecast data from netCDF files"""

    def load_dataset_in_memory(path):
        with xr.open_dataset(path) as dataset:
            return dataset.load()

    hist_ds = load_dataset_in_memory(historical_file)

    def open_or_build_mean(mean_path, min_path, max_path, label):
        mean_path = Path(mean_path)
        min_path = Path(min_path)
        max_path = Path(max_path)

        if mean_path.exists():
            return load_dataset_in_memory(mean_path)

        print(f"Mean forecast file missing for {label}: {mean_path}")
        print(f"Building {label} mean forecast from min/max files instead.")

        min_ds = load_dataset_in_memory(min_path)
        max_ds = load_dataset_in_memory(max_path)
        mean_ds = (min_ds + max_ds) / 2
        return mean_ds

    forecast_short_min = load_dataset_in_memory(forecast_files['short_min'])
    forecast_short_max = load_dataset_in_memory(forecast_files['short_max'])
    forecast_short_mean = open_or_build_mean(
        forecast_files['short_mean'],
        forecast_files['short_min'],
        forecast_files['short_max'],
        'short-range'
    )
    forecast_long_min = load_dataset_in_memory(forecast_files['long_min'])
    forecast_long_max = load_dataset_in_memory(forecast_files['long_max'])
    forecast_long_mean = open_or_build_mean(
        forecast_files['long_mean'],
        forecast_files['long_min'],
        forecast_files['long_max'],
        'long-range'
    )
    
    return hist_ds, {
        'short': (forecast_short_min, forecast_short_mean, forecast_short_max),
        'long': (forecast_long_min, forecast_long_mean, forecast_long_max)
    }

def load_climatology_for_location(climatology_file, city_locations):
    """Load climatology data for specific locations"""
    print(f"Loading climatology from: {climatology_file}")
    clim_ds = xr.open_dataset(climatology_file)
    city_climatology = {}
    
    print("Available coordinates:")
    print(f"Latitude range: {clim_ds.latitude.values.min():.2f} to {clim_ds.latitude.values.max():.2f}")
    print(f"Longitude range: {clim_ds.longitude.values.min():.2f} to {clim_ds.longitude.values.max():.2f}")
    
    # For each city, get its climatology
    for city, location in city_locations.items():
        print(f"\nProcessing {city}...")
        
        # Find the nearest grid point to the city's location
        city_data = clim_ds['t_2m'].sel(
            latitude=location['latitude'],
            longitude=location['longitude'],
            method='nearest'
        ).load()  # Force load the data
        
        # Get the actual coordinates selected
        selected_lat = float(city_data.latitude)
        selected_lon = float(city_data.longitude)
        
        # Store the climatology data for this city
        city_climatology[city] = city_data.values
        
        # Print debug info
        print(f"Location: requested ({location['latitude']:.2f} deg N, {location['longitude']:.2f} deg E)")
        print(f"         selected  ({selected_lat:.2f} deg N, {selected_lon:.2f} deg E)")
        print(f"Data shape: {city_climatology[city].shape}")
        print(f"Sample values (first 5 days): {city_climatology[city][:5]}")
    
    return city_climatology

def create_country_plot(country, cities, historical_data, forecast_data, city_mapping, climatology_data, output_dir):
    """Create temperature plot for a country with color coding based on comparison to climatology"""
    # Filter cities to only those available in ALL datasets
    forecast_short = forecast_data['short']
    forecast_long = forecast_data['long']
    available_cities = [c for c in cities
                        if c in historical_data.data_vars
                        and c in forecast_short[0].data_vars
                        and c in forecast_short[1].data_vars
                        and c in forecast_short[2].data_vars
                        and c in forecast_long[0].data_vars
                        and c in forecast_long[1].data_vars
                        and c in forecast_long[2].data_vars]
    
    if not available_cities:
        print(f"  WARNING: No cities with data available for {country} – skipping plot")
        return
    
    if len(available_cities) < len(cities):
        print(f"  Note: Using {len(available_cities)}/{len(cities)} cities for {country}: {available_cities}")
    
    cities = available_cities
    
    # Convert timestamps to datetime
    hist_times = pd.to_datetime(historical_data.time.values)
    
    # Calculate weights based on population
    total_population = sum(city_mapping[city]['population'] for city in cities)
    weights = {city: city_mapping[city]['population'] / total_population for city in cities}
    
    # Calculate weighted temperatures for historical data
    hist_temp = np.zeros_like(historical_data[cities[0]].values)
    for city in cities:
        hist_temp += historical_data[city].values * weights[city]
    
    # Get forecast dates
    short_times = pd.to_datetime(forecast_short[1].time.values, unit='ns')
    long_times = pd.to_datetime(forecast_long[1].time.values, unit='ns')
    
    # Calculate weighted temperatures for forecasts
    short_min = np.zeros_like(forecast_short[0][cities[0]].values)
    short_mean = np.zeros_like(forecast_short[1][cities[0]].values)
    short_max = np.zeros_like(forecast_short[2][cities[0]].values)
    long_min = np.zeros_like(forecast_long[0][cities[0]].values)
    long_mean = np.zeros_like(forecast_long[1][cities[0]].values)
    long_max = np.zeros_like(forecast_long[2][cities[0]].values)
    
    for city in cities:
        short_min += forecast_short[0][city].values * weights[city]
        short_mean += forecast_short[1][city].values * weights[city]
        short_max += forecast_short[2][city].values * weights[city]
        long_min += forecast_long[0][city].values * weights[city]
        long_mean += forecast_long[1][city].values * weights[city]
        long_max += forecast_long[2][city].values * weights[city]
    
    # Create the plot
    plt.figure(figsize=(15, 8))
    
    # Get climatological normal dates
    short_end_date = short_times[-1]
    
    # Find indices of long-term forecast times that are after the short-term forecast
    long_times_idx = [i for i, t in enumerate(long_times) if t > short_end_date]
    
    # Create a list of all times for the plot
    hist_times_last_week = hist_times[-7:]
    all_dates = pd.to_datetime(list(hist_times_last_week) + list(short_times) + list(long_times[long_times_idx]))
    day_numbers = [d.timetuple().tm_yday for d in all_dates]
    
    # Calculate weighted climatology
    normal_temps = np.zeros(len(day_numbers))
    for i, day in enumerate(day_numbers):
        # Ensure we wrap around for leap years (day 366)
        day_idx = (day - 1) % 365
        for city in cities:
            city_clim = climatology_data[city]
            normal_temps[i] += city_clim[day_idx] * weights[city]
    
    # Plot climatological normal
    plt.plot(all_dates, normal_temps, color='gray', linestyle='--', label='Normal', alpha=0.7)
    
    # Define historical count (last 7 days)
    hist_count = min(7, len(hist_times_last_week))
    hist_data = hist_temp[-hist_count:]
    
    # Plot historical data segments with blue/red coloring like forecast data
    for i in range(hist_count-1):
        # Determine colors based on comparison to normal
        if hist_data[i] < normal_temps[i] and hist_data[i+1] < normal_temps[i+1]:
            # Both points below normal
            line_color = 'blue'
        elif hist_data[i] >= normal_temps[i] and hist_data[i+1] >= normal_temps[i+1]:
            # Both points above normal
            line_color = 'red'
        else:
            # Transition between above and below
            line_color = 'gray'
        
        # Plot the historical segment
        plt.plot([all_dates[i], all_dates[i+1]], 
                 [hist_data[i], hist_data[i+1]], 
                 color=line_color, linewidth=2.5)
    
    # Add historical data to legend (green line kept for legend only)
    plt.plot([], [], color='k', linewidth=2.5, label='Historical')
    
    # Prepare arrays for short-term forecast
    short_dates = all_dates[hist_count:hist_count+len(short_times)]
    
    # Split short-term forecast into segments by above/below normal
    for i in range(len(short_times)-1):
        idx = i
        idx_next = i + 1
        norm_idx = hist_count + i
        norm_idx_next = hist_count + i + 1
        
        if norm_idx >= len(normal_temps) or norm_idx_next >= len(normal_temps):
            continue
        
        # Current and next point status
        current_below = short_mean[idx] < normal_temps[norm_idx]
        next_below = short_mean[idx_next] < normal_temps[norm_idx_next]
        
        # Choose color based on comparison to normal
        if current_below and next_below:
            # Both points below normal
            line_color = 'blue'
            fill_color = 'blue'
            alpha = 0.2
        elif not current_below and not next_below:
            # Both points above normal
            line_color = 'red'
            fill_color = 'red'
            alpha = 0.2
        else:
            # Transition between above and below - use gray for the line
            line_color = 'gray'
            # For the fill color, use a mix of both or the average color
            if current_below:
                fill_color = 'blue'
            else:
                fill_color = 'red'
            alpha = 0.2
        
        # Plot line segment
        plt.plot([short_dates[idx], short_dates[idx_next]], 
                 [short_mean[idx], short_mean[idx_next]], 
                 color=line_color, linewidth=1.5)
        
        # Add shading for uncertainty
        plt.fill_between([short_dates[idx], short_dates[idx_next]], 
                        [short_min[idx], short_min[idx_next]], 
                        [short_max[idx], short_max[idx_next]], 
                        color=fill_color, alpha=alpha,
                        interpolate=True)
    
    # Plot long-term forecast starting from where short-term forecast ends
    long_dates = all_dates[hist_count+len(short_times):]
    
    # Add a vertical line to separate short-term and long-term forecasts
    if len(short_dates) > 0 and len(long_dates) > 0:
        # Get the y-axis limits to ensure the line spans the entire height
        ymin, ymax = plt.ylim()
        # Add vertical line at the transition point (last short-term date)
        plt.axvline(x=short_dates[-1], color='black', linestyle='-', linewidth=1.5, alpha=0.7)
        # Add text annotation for the vertical line
        plt.text(short_dates[-1], ymax, " Short | Long ", 
                horizontalalignment='center', verticalalignment='top',
                backgroundcolor='white', fontsize=9)
    
    if len(long_times_idx) > 0 and len(long_dates) > 0:
        for i in range(len(long_times_idx)-1):
            idx_long = long_times_idx[i]
            idx_long_next = long_times_idx[i+1]
            idx_plot = i
            idx_plot_next = i + 1
            
            if idx_plot >= len(long_dates) or idx_plot_next >= len(long_dates):
                continue
                
            norm_idx = hist_count + len(short_times) + idx_plot
            norm_idx_next = norm_idx + 1
            
            if norm_idx >= len(normal_temps) or norm_idx_next >= len(normal_temps):
                continue
            
            # Current and next point status
            current_below = long_mean[idx_long] < normal_temps[norm_idx]
            next_below = long_mean[idx_long_next] < normal_temps[norm_idx_next]
            
            # Choose color based on comparison to normal
            if current_below and next_below:
                # Both points below normal
                line_color = 'blue'
                fill_color = 'blue'
                alpha = 0.2
            elif not current_below and not next_below:
                # Both points above normal
                line_color = 'red'
                fill_color = 'red'
                alpha = 0.2
            else:
                # Transition between above and below - use gray for the line
                line_color = 'gray'
                if current_below:
                    fill_color = 'blue'
                else:
                    fill_color = 'red'
                alpha = 0.2
            
            # Plot line segment
            plt.plot([long_dates[idx_plot], long_dates[idx_plot_next]], 
                    [long_mean[idx_long], long_mean[idx_long_next]], 
                    color=line_color, linewidth=1.5)
            
            # Add shading for uncertainty
            plt.fill_between([long_dates[idx_plot], long_dates[idx_plot_next]], 
                            [long_min[idx_long], long_min[idx_long_next]], 
                            [long_max[idx_long], long_max[idx_long_next]], 
                            color=fill_color, alpha=alpha,
                            interpolate=True)
    
    # Add legend entries for forecasts
    plt.plot([], [], color='blue', linewidth=1.5, label='Forecast (Below Normal)')
    plt.plot([], [], color='red', linewidth=1.5, label='Forecast (Above Normal)')
    
    # Create title with cities included
    cities_str = ", ".join(cities)
    plt.title(f'Temperatures in {country} ({cities_str}) as of {datetime.now().strftime("%Y-%m-%d")}')
    plt.xlabel('Date')
    plt.ylabel('Temperature (deg C)')
    
    # Use only horizontal grid lines, no vertical lines
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.legend()
    
    # Format x-axis with more frequent ticks (every 5 days)
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    
    # Adjust margins
    plt.margins(x=0.02)
    
    # Save plot
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{country.replace(" ", "_").replace("(", "").replace(")", "")}_temperature.png', dpi=300)
    plt.close()


def main():
    today = datetime.now(pytz.UTC)
    
    # File paths
    historical_file = r'P:\QFA\TonyWeather\marta\LPG_REPORT\historical\historical_2026.nc'
    forecast_files = {
        'short_min': rf'P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_short_min_{today.strftime("%Y%m%d")}.nc',
        'short_mean': rf'P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_short_mean_{today.strftime("%Y%m%d")}.nc',
        'short_max': rf'P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_short_max_{today.strftime("%Y%m%d")}.nc',
        'long_min': rf'P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_min_{today.strftime("%Y%m%d")}.nc',
        'long_mean': rf'P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_mean_{today.strftime("%Y%m%d")}.nc',
        'long_max': rf'P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_max_{today.strftime("%Y%m%d")}.nc'
    }
    climatology_file = r'P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_2013_2023.nc'
    output_dir = 'temperature_plots'
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    historical_data, forecast_data = load_and_process_data(historical_file, forecast_files)
    city_locations = get_city_locations()
    climatology_data = load_climatology_for_location(climatology_file, city_locations)
    
    # Get city mapping and group by country
    city_mapping = get_city_country_mapping()
    country_cities = {}
    for city, info in city_mapping.items():
        country = info['country']
        if country not in country_cities:
            country_cities[country] = []
        country_cities[country].append(city)
    
    # Create plots for each country
    for country, cities in country_cities.items():
        try:
            create_country_plot(country, cities, historical_data, forecast_data, 
                              city_mapping, climatology_data, output_dir)
            print(f"Successfully created plot for {country}")
        except Exception as e:
            print(f"Error creating plot for {country}: {str(e)}")
            print(f"Country: {country}")
            print(f"Cities: {cities}")
            import traceback
            print(traceback.format_exc())

if __name__ == '__main__':
    main()

Mean forecast file missing for short-range: P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_short_mean_20260528.nc
Building short-range mean forecast from min/max files instead.
Mean forecast file missing for long-range: P:\QFA\TonyWeather\marta\LPG_REPORT\forecasts\forecast_mean_20260528.nc
Building long-range mean forecast from min/max files instead.
Loading climatology from: P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_2013_2023.nc
Available coordinates:
Latitude range: -90.00 to 90.00
Longitude range: -180.00 to 180.00

Processing Augsburg...
Location: requested (48.50 deg N, 11.00 deg E)
         selected  (48.50 deg N, 11.00 deg E)
Data shape: (365,)
Sample values (first 5 days): [2.04025934 1.91247281 1.87885271 1.99297985 2.10339466]

Processing Karlsruhe...
Location: requested (49.00 deg N, 8.50 deg E)
         selected  (49.00 deg N, 8.50 deg E)
Data shape: (365,)
Sample values (first 5 days): [4.36410633 4.26730347 4.21052995 4.2571855  4.32090288]

Processing M

# CDD

In [25]:
"""
Generate historical CDD CSVs for China sub-regions (North / Central / South)
from LOCAL ERA5 reanalysis data (no API downloads needed).
Reads the consolidated detrended file, extracts nearest 0.5° grid point per city,
computes CDD = max(T_mean - 18, 0), aggregates per sub-region with population weights.
Skips if all three CSVs already exist.
"""
import numpy as np, pandas as pd, xarray as xr
from pathlib import Path

# ── Sub-region definitions ──
CHINA_SUBREGIONS = {
    'China_North':   ['Harbin','Changchun','Shenyang','Hohhot','Urumqi','Xining','Lanzhou',
                      'Beijing','Tianjin','Shijiazhuang','Taiyuan','Yinchuan','Jinan','Xian'],
    'China_Central': ['Zhengzhou','Hefei','Nanchang','Wuhan','Changsha'],
    'China_South':   ['Nanjing','Hangzhou','Fuzhou','Guangzhou','Nanning','Haikou',
                      'Chongqing','Guiyang','Kunming','Chengdu','Shanghai'],
}

_POP = {
    'Harbin':30290000,'Changchun':23170000,'Shenyang':41550000,'Hohhot':23800000,
    'Urumqi':26230000,'Xining':5930000,'Lanzhou':24580000,'Beijing':21830000,
    'Tianjin':13640000,'Shijiazhuang':78780000,'Taiyuan':34460000,'Yinchuan':7290000,
    'Jinan':100800000,'Zhengzhou':97850000,'Xian':39520000,'Nanjing':85260000,
    'Hangzhou':66269999,'Hefei':61270000,'Fuzhou':41880000,'Nanchang':45280000,
    'Wuhan':58440000,'Changsha':66040000,'Guangzhou':127060000,'Nanning':50470000,
    'Haikou':10270000,'Chongqing':32130000,'Guiyang':38560000,'Kunming':46930000,
    'Chengdu':83470000,'Shanghai':24800000,
}

BASE_TEMP = 18.0
HIST_DIR = Path(r'P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country')
HIST_DIR.mkdir(parents=True, exist_ok=True)

ERA5_FILE = Path(r'P:\QFA\TonyWeather\CLIMATOLOGIES\temperature\ERA5\data_0.5x0.5\C_T2_1979-2024_detrended.nc')

needed = {r: cities for r, cities in CHINA_SUBREGIONS.items()
          if not (HIST_DIR / f'{r}_daily_cdd.csv').exists()}

if not needed:
    print('All China sub-region CSVs already exist – nothing to do.')
else:
    print(f'Generating CSVs for: {list(needed.keys())}')

    # Get city coordinates from the forecast locations already defined
    locs = get_forecast_locations()
    all_cities = sorted({c for cities in needed.values() for c in cities})

    # Open local ERA5 file and select 2000 onwards
    print(f'Opening {ERA5_FILE.name} ...')
    ds = xr.open_dataset(ERA5_FILE)
    ds = ds.sel(time=slice('2000-01-01', None))
    print(f'  Time range: {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}')
    print(f'  Shape: {dict(ds.dims)}')

    # Extract nearest grid point for each city
    city_dfs = {}
    for city in all_cities:
        if city not in locs:
            print(f'  Warning: {city} not in locations – skipping')
            continue
        lat = locs[city]['latitude']
        lon = locs[city]['longitude']
        # Select nearest grid point
        ts = ds['t_mean_2m_24h'].sel(latitude=lat, longitude=lon, method='nearest')
        df = ts.to_dataframe().reset_index()[['time', 't_mean_2m_24h']]
        df = df.rename(columns={'time': 'date', 't_mean_2m_24h': 'temp'}).dropna()
        df['CDD'] = (df['temp'] - BASE_TEMP).clip(lower=0)
        city_dfs[city] = df
    print(f'  Extracted {len(city_dfs)} cities')

    ds.close()

    # Aggregate per sub-region and save
    for region, cities in needed.items():
        available = [c for c in cities if c in city_dfs]
        if not available:
            print(f'  No data for {region} – skipping')
            continue

        merged = None
        for city in available:
            cdf = city_dfs[city][['date', 'CDD']].rename(columns={'CDD': city})
            merged = cdf if merged is None else merged.merge(cdf, on='date', how='outer')

        merged = merged.sort_values('date').reset_index(drop=True)

        pops = np.array([_POP.get(c, 1) for c in available], dtype=float)
        weights = pops / pops.sum()
        cdd_cols = merged[available].values
        merged['cdd'] = np.nansum(cdd_cols * weights[None, :], axis=1)
        merged['dayofyear'] = merged['date'].dt.dayofyear

        out = merged[['date', 'dayofyear', 'cdd']]
        out_path = HIST_DIR / f'{region}_daily_cdd.csv'
        out.to_csv(out_path, index=False)
        print(f'  Saved {out_path.name}  ({len(out):,} rows, {len(available)} cities)')

    print('\nDone generating China sub-region historical CSVs.')

All China sub-region CSVs already exist – nothing to do.


In [26]:
import pandas as pd
import numpy as np
import xarray as xr
import pytz
from datetime import datetime, timedelta
from pathlib import Path
import logging
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Configuration & Logging
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

ARROW = "->"

tz = pytz.timezone("Europe/Zurich")
today = datetime.now(tz)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Paths
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
base     = Path(r'P:/QFA/TonyWeather/marta/LPG_REPORT')
hist_dir = base / 'CDD_history_by_country'
plot_dir = hist_dir / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

nc2025   = base / 'historical' / 'historical_2026.nc'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Region definitions & population
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGION_MAP = {
    'Brazil':              ['Curitiba', 'Suape', 'Santos'],
    'China_North':         ['Harbin', 'Changchun', 'Shenyang', 'Hohhot', 'Urumqi', 'Xining', 'Lanzhou',
                            'Beijing', 'Tianjin', 'Shijiazhuang', 'Taiyuan', 'Yinchuan', 'Jinan', 'Xian'],
    'China_Central':       ['Zhengzhou', 'Hefei', 'Nanchang', 'Wuhan', 'Changsha'],
    'China_South':         ['Nanjing', 'Hangzhou', 'Fuzhou', 'Guangzhou', 'Nanning', 'Haikou',
                            'Chongqing', 'Guiyang', 'Kunming', 'Chengdu', 'Shanghai'],
    'France':              ['Bergerac', 'Le Mans'],
    'Germany':             ['Augsburg', 'Karlsruhe', 'Mannheim'],
    'India':               ['Delhi', 'Jamnagar', 'Haldia'],
    'Morocco':             ['Casablanca'],
    'Panama':              ['Panama'],
    'Japan':               ['Niigata', 'Sendai', 'Yamaguchi', 'Kobe', 'Oita', 'Aomori'],
    'South_Korea':         ['Seoul', 'Busan', 'Yeosu'],
    'Columbia':            ['Columbia'],
    'Tallahassee':         ['Tallahassee'],
    'Raleigh':             ['Raleigh'],
    'Kansas_and_Oklahoma': ['Kansas-City', 'Oklahoma-City'],
}

POP_MAP = {
    'Augsburg':300000, 'Karlsruhe':310000, 'Mannheim':320000,
    'Bergerac':27000,  'Le Mans':143000,
    'Delhi':19000000,  'Jamnagar':600000,  'Haldia':200000,
    'Casablanca':3500000,
    'Curitiba':1900000, 'Suape':30000, 'Santos':430000,
    'Panama':880000,
    'Niigata':800000,  'Sendai':1000000, 'Yamaguchi':145000,
    'Kobe':1500000,    'Oita':470000,     'Aomori':280000,
    'Seoul':9700000,   'Busan':3400000,   'Yeosu':300000,
    'Harbin':30290000,
    'Changchun':23170000,
    'Shenyang':41550000,
    'Hohhot':23800000,
    'Urumqi':26230000,
    'Xining':5930000,
    'Lanzhou':24580000,
    'Beijing':21830000,
    'Tianjin':13640000,
    'Shijiazhuang':78780000,
    'Taiyuan':34460000,
    'Yinchuan':7290000,
    'Jinan':100800000,
    'Zhengzhou':97850000,
    'Xian':39520000,
    'Nanjing':85260000,
    'Hangzhou':66269999,
    'Hefei':61270000,
    'Fuzhou':41880000,
    'Nanchang':45280000,
    'Wuhan':58440000,
    'Changsha':66040000,
    'Guangzhou':127060000,
    'Nanning':50470000,
    'Haikou':10270000,
    'Chongqing':32130000,
    'Guiyang':38560000,
    'Kunming':46930000,
    'Chengdu':83470000,
    'Shanghai':24800000,
    'Kansas-City':510000, 'Oklahoma-City':650000,
    'Columbia':133000, 'Tallahassee':194000, 'Raleigh':480000
}

# Reverse lookup
city_to_region = {city: region for region, cities in REGION_MAP.items() for city in cities}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1) Load historical region-level CSVs (2000-2024)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_historical_region(directory: Path) -> pd.DataFrame:
    """Combine all *_daily_cdd.csv into one DataFrame of region-level CDD."""
    rows = []
    for csv in directory.glob('*_daily_*.csv'):
        stem_l = csv.stem.lower()
        if not stem_l.endswith('_daily_cdd'):
            continue

        region = csv.stem[:-len('_daily_cdd')]
        df = pd.read_csv(csv, parse_dates=['date'])
        df.columns = [c.strip().lower() for c in df.columns]

        if 'cdd' not in df.columns:
            logger.warning(f"Skipping {csv.name}: missing 'cdd' column. Found: {list(df.columns)}")
            continue

        df['date'] = df['date'].dt.tz_localize(tz)
        df = df[df['date'].dt.year < 2026]
        df['region'] = region
        df = df.rename(columns={'cdd': 'CDD'})
        rows.append(df[['date','region','CDD']])

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['date','region','CDD'])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2) Load & aggregate 2025/2026 NetCDF by region (convert °C → CDD)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_netcdf_2025(nc_path: Path) -> pd.DataFrame:
    if not nc_path.exists():
        logger.warning(f"NetCDF file not found: {nc_path}")
        return pd.DataFrame()
    logger.info("Loading 2025 NetCDF ...")
    with xr.open_dataset(nc_path, decode_times=False) as ds:
        base_date = ds.time.attrs['units'].split(' since ')[1]
        idx = pd.date_range(start=base_date, periods=ds.dims['time'], freq='D', tz=tz)

        df = (
            ds.assign_coords(time=idx)
              .to_dataframe()
              .reset_index()
              .melt(id_vars=['time'], var_name='city', value_name='temp')
              .rename(columns={'time':'date'})
        )

    BASE = 18.0
    df['CDD'] = (df.temp - BASE).clip(lower=0)
    df = df.drop(columns='temp')

    df['region']     = df.city.map(city_to_region)
    df['population'] = df.city.map(POP_MAP)
    df = df.dropna(subset=['region','population'])

    df = (
        df.groupby(['date','region'])
          .apply(lambda g: np.average(g.CDD, weights=g.population))
          .reset_index(name='CDD')
    )
    return df


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2b) Supplement current-year CDD from local ERA5 for China sub-regions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_era5_current_year_china() -> pd.DataFrame:
    """Load 2025+2026 ERA5 data for ALL China cities and compute region-level CDD.
    This supplements the mix-obs NetCDF which may be missing some Chinese cities."""
    era5_dir = Path(r'P:\QFA\TonyWeather\CLIMATOLOGIES\temperature\ERA5\data_0.5x0.5')
    files_to_load = []
    for fname in ['C_T2_2025_detrended.nc', 'C_T2_2026_detrended.nc']:
        p = era5_dir / fname
        if p.exists():
            files_to_load.append(p)

    if not files_to_load:
        return pd.DataFrame(columns=['date', 'region', 'CDD'])

    locs = get_forecast_locations()
    china_regions = ['China_North', 'China_Central', 'China_South']
    china_cities = []
    for r in china_regions:
        china_cities.extend(REGION_MAP.get(r, []))

    rows = []
    for fpath in files_to_load:
        ds = xr.open_dataset(fpath)
        for city in china_cities:
            if city not in locs:
                continue
            lat = locs[city]['latitude']
            lon = locs[city]['longitude']
            ts = ds['t_mean_2m_24h'].sel(latitude=lat, longitude=lon, method='nearest')
            vals = ts.values
            times = pd.to_datetime(ts.time.values)
            for t, v in zip(times, vals):
                if not np.isnan(v):
                    rows.append({'date': t, 'city': city, 'temp': v})
        ds.close()

    if not rows:
        return pd.DataFrame(columns=['date', 'region', 'CDD'])

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(tz)
    df['CDD'] = (df['temp'] - 18.0).clip(lower=0)
    df['region'] = df['city'].map(city_to_region)
    df['population'] = df['city'].map(POP_MAP)
    df = df.dropna(subset=['region', 'population'])

    df = (
        df.groupby(['date', 'region'])
          .apply(lambda g: np.average(g.CDD, weights=g.population))
          .reset_index(name='CDD')
    )
    # Only keep China sub-regions
    df = df[df['region'].isin(china_regions)]
    logger.info(f"ERA5 China supplement: {len(df):,} rows, regions: {sorted(df.region.unique())}")
    return df


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3) CDD window helper
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def get_CDD_window(
    df: pd.DataFrame,
    year: int,
    full: bool = True,
    cutoff_date: pd.Timestamp = None
) -> pd.DataFrame:
    """
    Return daily & cumulative CDD for the cooling season.
    Season: Apr 15 - Sep 30 (same year).
    """
    start = pd.Timestamp(f"{year}-04-15", tz=tz)
    end   = pd.Timestamp(f"{year}-10-01", tz=tz)

    if cutoff_date is None:
        cutoff = min(today, end - timedelta(days=1))
    else:
        cutoff = min(cutoff_date, end - timedelta(days=1))

    daily = (
        df[(df.date >= start) & (df.date <= cutoff)]
          .groupby('date', as_index=False)['CDD']
          .sum()
    )

    if full:
        idx = pd.date_range(start, end - timedelta(days=1), tz=tz)
        daily = (
            daily.set_index('date')
                 .reindex(idx, fill_value=0)
                 .rename_axis('date')
                 .reset_index()
        )

    daily['days']    = (daily.date - start).dt.days
    daily['cum_CDD'] = daily.CDD.cumsum()
    return daily


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4) Similarity functions (with cutoff support)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def weighted_rmse(a, b):
    w = np.linspace(1, 2, len(a)); w /= w.mean()
    return np.sqrt((w * (a - b) ** 2).sum() / w.sum())


def similarity_score(cur, hist):
    rmse = np.sqrt(((cur - hist) ** 2).mean())
    wrmse = weighted_rmse(cur, hist)
    roc = np.sqrt(((np.gradient(cur) - np.gradient(hist)) ** 2).mean()) if len(cur) > 1 else 0
    corr = pearsonr(cur, hist)[0] if cur.std() > 0 and hist.std() > 0 else 0
    tl = min(30, len(cur))
    tailc = pearsonr(cur[-tl:], hist[-tl:])[0] if tl > 1 else 0
    cerr = 1 - abs(corr)
    terr = 1 - abs(tailc)
    w = {"rmse": 0.2, "wrmse": 0.25, "roc": 0.25, "cerr": 0.1, "terr": 0.2}
    return (w['rmse'] * rmse + w['wrmse'] * wrmse + w['roc'] * roc
            + w['cerr'] * cerr + w['terr'] * terr)


def find_similars(
    df: pd.DataFrame,
    year: int,
    start_year: int = 2000,
    n: int = 5,
    cutoff_date: pd.Timestamp = None
) -> list:
    """Compute the n most-similar past years based on cumulative CDD."""
    cur = get_CDD_window(df, year, full=False, cutoff_date=cutoff_date).cum_CDD.values
    logger.info(f"Finding {n} similar years to {year} (current has {len(cur)} days)")

    best = []
    for y in sorted(df.date.dt.year.unique()):
        if not (start_year <= y < year):
            continue
        hist = get_CDD_window(df, y, full=False, cutoff_date=cutoff_date).cum_CDD.values
        if hist.size == 0:
            continue
        L = min(len(cur), len(hist))
        score = similarity_score(cur[:L], hist[:L])
        best.append((y, score))

    similar_years = [y for y, _ in sorted(best, key=lambda t: t[1])[:n]]
    logger.info(f"Most similar years: {similar_years}")
    return similar_years


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5) Plot comparison
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def plot_comparison(df, year, similars, start_year=2000):
    cur = get_CDD_window(df, year, full=False).rename(columns={'cum_CDD':'cur'})
    fig, (ax1, ax2) = plt.subplots(2,1, figsize=(11,7), sharex=True)

    for y in sorted(df.date.dt.year.unique()):
        if start_year <= y < year and y not in similars:
            h = get_CDD_window(df, y, full=True)
            ax1.plot(h.days, h.cum_CDD, color='grey', alpha=0.08)

    ax1.plot(cur.days, cur.cur, color='green', lw=3.5, label=str(year))

    palette = ['blue','purple','orange','brown','cyan']
    for y,c in zip(similars, palette):
        h = get_CDD_window(df, y, full=True)
        ax1.plot(h.days, h.cum_CDD, color=c, lw=1, label=str(y))
        merged = cur.merge(
            h[['days','cum_CDD']].rename(columns={'cum_CDD':'hist_cum'}),
            on='days'
        )
        ax2.plot(merged['days'], merged['hist_cum']-merged['cur'], color=c, lw=1.8)

    ax2.axhline(0, ls='--', color='k', alpha=0.4)
    if ax2.lines:
        diffs = np.concatenate([ln.get_ydata() for ln in ax2.lines])
        ax2.axhspan(-diffs.std(), diffs.std(), alpha=0.1, color='grey')

    for ax in (ax1, ax2):
        ax.grid(True, alpha=0.25)
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            ax.legend(bbox_to_anchor=(1,1), loc='upper left', fontsize=8)

    ax1.set_ylabel('Cumulative CDD')
    ax2.set_ylabel('Historic - Current')
    ax2.set_xlabel('Days since start of cooling season')
    fig.suptitle(f"{df.region.iloc[0]} - CDD trajectory {year}")
    fig.subplots_adjust(right=0.82, hspace=0.25)
    return fig

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6) Main driver
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def main():
    hist_df = load_historical_region(hist_dir)
    logger.info(f"Loaded historical CSVs {ARROW} {len(hist_df):,} rows")

    cur_df  = load_netcdf_2025(nc2025)
    logger.info(f"2025 NetCDF aggregated {ARROW} {len(cur_df):,} rows")

    # Supplement with local ERA5 for China sub-regions (covers missing cities in mix-obs)
    era5_china_df = load_era5_current_year_china()

    full_df = pd.concat([hist_df, cur_df, era5_china_df], ignore_index=True)
    # For China regions, prefer ERA5 supplement (has all cities); drop duplicates keeping last
    full_df = full_df.drop_duplicates(subset=['date', 'region'], keep='last')
    full_df = full_df.sort_values(['region', 'date']).reset_index(drop=True)
    logger.info(f"Combined dataset {ARROW} {len(full_df):,} rows")

    years_available = sorted(full_df.date.dt.year.unique())
    logger.info(f"Years available: {years_available[0]}-{years_available[-1]} ({len(years_available)} years)")

    for region in sorted(full_df.region.unique()):
        logger.info(f"\nProcessing {region}...")
        df_region = full_df[full_df.region == region]

        # Cooling season: mid-April through September of the same year.
        season_start = pd.Timestamp(f"{today.year}-04-15", tz=tz)
        season_end   = pd.Timestamp(f"{today.year}-10-01", tz=tz)

        window_data = df_region[(df_region.date >= season_start) & (df_region.date <= min(today, season_end))]
        if window_data.empty:
            logger.info(f"Skipping {region} - no data in CDD season window")
            continue

        logger.info(f"  {len(window_data)} days of {today.year} CDD season data")
        try:
            last_obs = window_data.date.max()
            similars = find_similars(df_region, today.year, cutoff_date=last_obs)

            fig = plot_comparison(df_region, today.year, similars)
            out_png = plot_dir / f"{region}_summer_CDD_comparison.png"
            fig.savefig(out_png, dpi=110, bbox_inches="tight")
            plt.close(fig)
            logger.info(f"Saved {ARROW} {out_png}")
        except Exception as e:
            logger.error(f"Error processing {region}: {e}")
            import traceback; traceback.print_exc()


if __name__ == "__main__":
    main()

2026-05-28 13:14:14,589 - INFO - Loaded historical CSVs -> 166,815 rows
2026-05-28 13:14:14,610 - INFO - Loading 2025 NetCDF ...
C:\Users\LORECOMI\AppData\Local\Temp\ipykernel_55808\319762718.py:144: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  idx = pd.date_range(start=base_date, periods=ds.dims['time'], freq='D', tz=tz)
C:\Users\LORECOMI\AppData\Local\Temp\ipykernel_55808\319762718.py:164: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(g.CDD, weights=g.population))
2026-05-

## SECTION PADDS CDD

In [27]:
import subprocess
import sys
import os
from pathlib import Path
# ========================================
# SECTION 0: RUN DATA GENERATION SCRIPTS
# ========================================
print("="*60)
print("STEP 0: Running PADD data generation and plotting scripts")
print("="*60)

# Define the folder containing all scripts
scripts_folder = Path(r"P:\QFA\TonyWeather\marta\LPG_REPORT\PADD_plots")

# Define the scripts to run in order
scripts_to_run = [
    #"US_download_data_Ridi_V0.94.py",  # Downloads and processes CDD data
    #"download_data.py",                 # Downloads forecast data
    'download_historical_and_forecast_CORRECTED.py',
    "plot_PADD_CDD_IMPROVED.py"                  # Generates the plots
]

# Run each script sequentially
for script_name in scripts_to_run:
    script_path = scripts_folder / script_name
    
    if not script_path.exists():
        print(f"\nâš ï¸  WARNING: Script not found: {script_path}")
        print(f"   Skipping {script_name}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Running: {script_name}")
    print(f"{'='*60}\n")
    
    try:
        # Run the script using subprocess
        result = subprocess.run(
            [sys.executable, str(script_path)],
            cwd=str(scripts_folder),  # Set working directory
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour timeout
        )
        
        # Print output
        if result.stdout:
            print(result.stdout)
        
        # Check for errors
        if result.returncode != 0:
            print(f"\nâŒ ERROR: {script_name} failed with return code {result.returncode}")
            if result.stderr:
                print(f"Error details:\n{result.stderr}")
            print(f"\nâš ï¸  Continuing with PDF generation anyway...")
        else:
            print(f"\nâœ… {script_name} completed successfully")
            
    except subprocess.TimeoutExpired:
        print(f"\nâ±ï¸  TIMEOUT: {script_name} exceeded 1 hour execution time")
        print(f"   Continuing with PDF generation...")
    except Exception as e:
        print(f"\nâŒ EXCEPTION while running {script_name}: {str(e)}")
        print(f"   Continuing with PDF generation...")

print(f"\n{'='*60}")
print("All PADD scripts completed. Starting PDF generation...")
print(f"{'='*60}\n")

STEP 0: Running PADD data generation and plotting scripts

Running: download_historical_and_forecast_CORRECTED.py


=== Creating Directory Structure ===
✅ raw_netcdf: OK
✅ daily_hdd_cdd: OK
✅ HDD_combined: OK
✅ PADDS: OK


=== Unified FAST HDD/CDD Pipeline Started ===
Creating PADD masks (FAST)...
  PADD1A: 85 grid points
  PADD1B: 142 grid points
  PADD1C: 276 grid points
  PADD2: 1114 grid points
  PADD3: 581 grid points
  PADD4: 592 grid points
  PADD5: 1859 grid points
  PADD1 combined: 503

Processing ERA5 winter: 2023-10-01 → 2024-03-31
ERA5 shape: 183 timesteps, 53 lats, 119 lons
ERA5 lat range: [24. 50.]
ERA5 lon range: [-125.  -66.]

  Processing PADD1A...
    Mask lat range: [-90.  90.]
    Mask lon range: [-180.  180.]
    Valid mask pixels: 85
    Temperature stats:
      Min: -11.88
      Max: 18.15
      Mean: 1.57
      First 3 values: [13.477556 14.540338 15.319146]
    Generated 183 rows
    ✓ Saved to PADD1A_daily_hdd_combined_updated.csv

  Processing PADD1B...
    M

# MAPS

In [28]:
import time
import os
import pytz
import netCDF4
import requests
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
from msal import ConfidentialClientApplication
import warnings
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

start_time = time.time()
DSL_V3_METEO_URL = "https://fs_gateway.prod.axponet.ch/dsl/v3/meteo"

# ------------------------------------------------------------------------------
# Authentication
# ------------------------------------------------------------------------------
def dsl_authentication(app_id="5a838d41-4163-42ef-8479-48fe7db6d547", env_name="MDS_SECRET"):
    axpo_azure_tenant_id = "8619c67c-945a-48ae-8e77-35b1b71c9b98"
    app_authority_url = "https://login.microsoftonline.com/" + axpo_azure_tenant_id
    dsl_api_app_read_scope = "api://e504750d-ac05-4807-a2e7-8fcd77b247aa/.default"

    client_secret = os.getenv(env_name)
    if not client_secret:
        raise Exception(f"Environment variable {env_name} not found. Restart kernel after setting it.")

    auth_app = ConfidentialClientApplication(
        client_id=app_id,
        authority=app_authority_url,
        client_credential=client_secret
    )
    result = auth_app.acquire_token_for_client(scopes=[dsl_api_app_read_scope])
    if "access_token" not in result:
        raise Exception(f"Unable to fetch token: {result.get('error_description', 'Unknown')}")

    return {
        "Authorization": f"Bearer {result['access_token']}",
        "User-Agent": os.environ.get('USER', os.environ.get('USERNAME', 'axpo-user')),
        "Accept-Encoding": "gzip, deflate",
        "Content-Type": "application/json"
    }

# ------------------------------------------------------------------------------
# Helper: get a writable save directory (with fallback to local Desktop)
# ------------------------------------------------------------------------------
def get_writable_dir(preferred_dir):
    """
    Try to use preferred_dir. If not writable, fall back to Desktop.
    Returns the path that is actually writable.
    """
    for candidate in [preferred_dir, os.path.join(os.path.expanduser("~"), "Desktop", "SEAS_output")]:
        try:
            os.makedirs(candidate, exist_ok=True)
            test = os.path.join(candidate, '_write_test.tmp')
            with open(test, 'w') as f:
                f.write('ok')
            os.remove(test)
            if candidate != preferred_dir:
                print(f"WARNING: Preferred path not writable: {preferred_dir}")
                print(f"         Falling back to            : {candidate}")
            else:
                print(f"Save directory OK: {candidate}")
            return candidate
        except PermissionError:
            continue

    raise PermissionError(
        f"Cannot write to any location!\n"
        f"  Tried: {preferred_dir}\n"
        f"  Tried: Desktop fallback\n"
        f"  Please check your network drive connection (VPN?) and permissions."
    )


def save_bytes_safely(content, target_path, fallback_dir=None):
    target_dir = os.path.dirname(target_path)
    file_name = os.path.basename(target_path)
    stem, ext = os.path.splitext(file_name)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    directories_to_try = [target_dir]
    if fallback_dir and os.path.normcase(fallback_dir) != os.path.normcase(target_dir):
        directories_to_try.append(fallback_dir)

    errors = []

    for directory in directories_to_try:
        os.makedirs(directory, exist_ok=True)
        candidate_names = [
            file_name,
            f"{stem}_{timestamp}{ext}"
        ]

        for candidate_name in candidate_names:
            candidate_path = os.path.join(directory, candidate_name)
            temp_path = candidate_path + ".tmp"

            try:
                with open(temp_path, 'wb') as f:
                    f.write(content)

                if os.path.exists(candidate_path):
                    try:
                        os.remove(candidate_path)
                    except PermissionError as exc:
                        errors.append((candidate_path, str(exc)))
                        if os.path.exists(temp_path):
                            os.remove(temp_path)
                        continue

                os.replace(temp_path, candidate_path)
                return candidate_path

            except PermissionError as exc:
                errors.append((candidate_path, str(exc)))
                if os.path.exists(temp_path):
                    os.remove(temp_path)
                continue

    formatted_errors = "\n".join(f"  {path}: {error}" for path, error in errors)
    raise PermissionError(
        "Unable to save the NetCDF file because the target file or directory is locked.\n"
        f"Tried:\n{formatted_errors}"
    )

# ------------------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------------------
date_format = "%Y-%m-%dT%H:%M:%SZ"
start_date = datetime.now(pytz.UTC)
forecast_days = 150
timeout = 9000

lons = np.arange(-180, 180, 1)
lats = np.arange(0, 90, 1)

start_date_str = start_date.strftime("%Y-%m-%d")
end_date = start_date + timedelta(days=forecast_days)
location = f"{lats[-1]},{lons[0]}_{lats[0]},{lons[-1]}:1,1"
time_int = f"{start_date.strftime(date_format)}--{end_date.strftime(date_format)}:P1D"

# ------------------------------------------------------------------------------
# Save directory with automatic fallback
# ------------------------------------------------------------------------------
preferred_dir = r'P:\QFA\TonyWeather\marta\LPG_REPORT\SEAS'
save_dir = get_writable_dir(preferred_dir)
desktop_fallback_dir = os.path.join(os.path.expanduser("~"), "Desktop", "SEAS_output")

# ------------------------------------------------------------------------------
# Authenticate
# ------------------------------------------------------------------------------
print('\nAuthenticating with DSL API...')
headers = dsl_authentication()
print('Authentication successful\n')

# ------------------------------------------------------------------------------
# Build DSL v3 payload
# ------------------------------------------------------------------------------
payload = {
    "parameters": "t_mean_2m_24h:K",
    "location": location,
    "source": "ecmwf-mmsf",
    "datetime": time_int,
    "outputFormat": "netcdf"
}

print(f"Requesting endpoint:\n{DSL_V3_METEO_URL}\n")
print(f"Payload:\n{payload}\n")

# ------------------------------------------------------------------------------
# Download
# ------------------------------------------------------------------------------
raw_response = requests.post(DSL_V3_METEO_URL, json=payload, headers=headers, verify=False, timeout=timeout)

if 'Warning' in raw_response.headers:
    print(f"WARNING: Deprecation warning: {raw_response.headers['Warning']}")
    print(f"         Sunset date         : {raw_response.headers.get('Sunset', 'unknown')}\n")

if raw_response.status_code == 200:
    first_bytes = raw_response.content[:20]
    is_netcdf = first_bytes.startswith(b'CDF') or first_bytes.startswith(b'\x89HDF')

    print(f"Status      : {raw_response.status_code}")
    print(f"Content-Type: {raw_response.headers.get('Content-Type')}")
    print(f"Size        : {len(raw_response.content):,} bytes")
    print(f"Is NetCDF   : {'Yes' if is_netcdf else 'No'}")

    if not is_netcdf:
        print(f"\nResponse preview: {raw_response.text[:300]}")
    else:
        ds = xr.open_dataset(
            xr.backends.NetCDF4DataStore(
                netCDF4.Dataset('in-mem-file', mode='r', memory=raw_response.content)
            )
        )
        print(f"\n{ds}\n")

        target_filename = os.path.join(save_dir, f'temperature_24h_SEAS_{start_date_str}.nc')
        saved_path = save_bytes_safely(
            raw_response.content,
            target_filename,
            fallback_dir=desktop_fallback_dir
        )
        print(f"Saved to: {saved_path}")
else:
    print(f"Failed: HTTP {raw_response.status_code}")
    print(f"Response: {raw_response.text[:500]}")

# ------------------------------------------------------------------------------
# Timing
# ------------------------------------------------------------------------------
print(f"\nThe code took {time.time() - start_time:.2f} seconds to execute.")


Save directory OK: P:\QFA\TonyWeather\marta\LPG_REPORT\SEAS

Authenticating with DSL API...
Authentication successful

Requesting endpoint:
https://fs_gateway.prod.axponet.ch/dsl/v3/meteo

Payload:
{'parameters': 't_mean_2m_24h:K', 'location': '89,-180_0,179:1,1', 'source': 'ecmwf-mmsf', 'datetime': '2026-05-28T11:18:56Z--2026-10-25T11:18:56Z:P1D', 'outputFormat': 'netcdf'}

Status      : 200
Content-Type: application/netcdf
Size        : 12,699,237 bytes
Is NetCDF   : Yes

<xarray.Dataset> Size: 39MB
Dimensions:        (time: 151, latitude: 90, longitude: 360)
Coordinates:
  * time           (time) datetime64[ns] 1kB 2026-05-28T11:18:56 ... 2026-10-...
  * latitude       (latitude) float64 720B 0.0 1.0 2.0 3.0 ... 87.0 88.0 89.0
  * longitude      (longitude) float64 3kB -180.0 -179.0 -178.0 ... 178.0 179.0
Data variables:
    crs_wgs84      int64 8B ...
    t_mean_2m_24h  (time, latitude, longitude) float64 39MB ...
Attributes:
    Service:      Meteomatics Weather API
    Website:  

In [29]:
import xarray as xr
import numpy as np
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import glob
from datetime import datetime, timedelta, date

# Set start date
start_date = date.today().strftime('%Y-%m-%d')
start_date2 = str(start_date)

# Define file paths
anomalies_file = rf'P:\QFA\TonyWeather\marta\LPG_REPORT\out_files\t_mean_2m_24h_anom_{start_date2}.nc'
monthly_means_file = rf'P:\QFA\TonyWeather\marta\LPG_REPORT\out_files\t_mean_2m_24h_anom_monthly_means_{start_date2}.nc'

# Updated climatology path
climatology_file = r'P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_1993_2023.nc'

# Function to print diagnostic statistics
def print_stats(data, name):
    """Print diagnostic statistics for a dataset"""
    if hasattr(data, 'values'):
        values = data.values
    else:
        values = data
    
    print(f"\n--- {name} Statistics ---")
    print(f"Shape: {values.shape}")
    print(f"Data type: {values.dtype}")
    print(f"Min: {np.nanmin(values)}")
    print(f"Max: {np.nanmax(values)}")
    print(f"Mean: {np.nanmean(values)}")
    print(f"Median: {np.nanmedian(values)}")
    print(f"Has NaN: {np.isnan(values).any()}")
    print(f"NaN count: {np.isnan(values).sum()}")
    print("------------------------\n")

# Part 1: Calculate and Save Anomalies
if not os.path.exists(anomalies_file) or True:  # Force recalculation
    # Resolve forecast file path robustly (exact-date file or latest fallback/timestamped file)
    forecast_dir = r'P:\QFA\TonyWeather\marta\LPG_REPORT\SEAS'
    expected_forecast_file = os.path.join(forecast_dir, f'temperature_24h_SEAS_{start_date2}.nc')

    if os.path.exists(expected_forecast_file):
        forecast_file_to_open = expected_forecast_file
    else:
        candidates = sorted(
            glob.glob(os.path.join(forecast_dir, 'temperature_24h_SEAS_*.nc')),
            key=os.path.getmtime,
            reverse=True
        )
        if not candidates:
            raise FileNotFoundError(
                f"No forecast file found in {forecast_dir}. "
                "Run the previous forecast-download cell first."
            )
        forecast_file_to_open = candidates[0]
        print(f"Expected file not found for today: {expected_forecast_file}")
        print(f"Using latest available forecast file: {forecast_file_to_open}")

    # Load the daily forecast data
    try:
        daily_data = xr.open_dataset(forecast_file_to_open, engine='netcdf4')
    except Exception as e:
        print(f"netcdf4 engine failed ({e}), trying scipy engine...")
        daily_data = xr.open_dataset(forecast_file_to_open, engine='scipy')

    print("Loaded daily forecast data")
    
    # Print basic info about forecast data
    print("\nForecast data variables:", list(daily_data.data_vars))
    print("Forecast data dimensions:", daily_data.dims)
    print("Forecast data time range:", daily_data.time.values[0], "to", daily_data.time.values[-1])
    
    print_stats(daily_data['t_mean_2m_24h'], "Forecast Temperature")
    
    # Check if values seem to be in Kelvin (if typical values are > 100)
    forecast_vals = daily_data['t_mean_2m_24h'].values
    if np.nanmean(forecast_vals) > 100:
        print("WARNING: Forecast data seems to be in Kelvin (mean > 100)")
        print("Converting forecast data from Kelvin to Celsius...")
        daily_data['t_mean_2m_24h'] = daily_data['t_mean_2m_24h'] - 273.15
        print_stats(daily_data['t_mean_2m_24h'], "Forecast Temperature (after K->C conversion)")
    
    # Check if climatology file exists
    if not os.path.exists(climatology_file):
        print(f"Warning: Climatology file not found at: {climatology_file}")
        print("Searching for alternative climatology files...")
        
        # Try alternative locations
        possible_paths = [
            r'P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_detrended_1993_2023_renamed.nc'
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                climatology_file = path
                print(f"Found climatology file at: {climatology_file}")
                break
        
        if not os.path.exists(climatology_file):
            raise FileNotFoundError(f"Could not find climatology file in any expected location")
    
    # Load the climatology data
    print(f"Loading climatology from: {climatology_file}")
    climatology = xr.open_dataset(climatology_file)
    print("Loaded climatology data")
    
    # Fix implemented by Martin on 22nd October 2025
    clim_doy = climatology['dayofyear'].values.copy()
    clim_doy_adj = np.where(clim_doy > 59, clim_doy - 1, clim_doy)
    climatology = climatology.assign_coords(dayofyear=clim_doy_adj)
    
    # Print basic info about climatology data
    print("\nClimatology data variables:", list(climatology.data_vars))
    print("Climatology data dimensions:", climatology.dims)
    
    print_stats(climatology['t_2m'], "Climatology Temperature")
    
    # Process coordinates and calculate anomalies
    if np.any(climatology.longitude > 180):
        print("Adjusting climatology longitude from 0-360 to -180-180")
        climatology = climatology.assign_coords(longitude=((climatology.longitude + 180) % 360) - 180)
    
    # Print a few sample points
    print("\nSample climatology values for different times of year:")
    for day in [1, 90, 180, 270, 364]:
        sample = climatology['t_2m'].isel(dayofyear=day, latitude=180, longitude=360)
        print(f"Day {day+1}, sample value: {sample.values}")
    
    # Remove duplicates if needed
    unique_lat = ~daily_data.latitude.to_index().duplicated()
    unique_lon = ~daily_data.longitude.to_index().duplicated()
    daily_data = daily_data.isel(latitude=unique_lat, longitude=unique_lon)
    
    unique_lat_clim = ~climatology.latitude.to_index().duplicated()
    unique_lon_clim = ~climatology.longitude.to_index().duplicated()
    climatology = climatology.isel(latitude=unique_lat_clim, longitude=unique_lon_clim)
    
    # Sort coordinates
    daily_data = daily_data.sortby(['latitude', 'longitude'])
    climatology = climatology.sortby(['latitude', 'longitude'])
    
    # Interpolate climatology to forecast grid
    day_of_year = daily_data['time'].dt.dayofyear
    
    # Create an empty dataset for the interpolated climatology
    clim_interp = np.zeros((len(daily_data.time), len(daily_data.latitude), len(daily_data.longitude)))
    
    # For each time point, interpolate the corresponding day from climatology
    for i, day in enumerate(day_of_year.values):
        if i % 30 == 0:
            print(f"Processing day {i+1}/{len(day_of_year)}, day of year: {day}")
            
        # Handle Feb 29 in leap years
        if day > 365:
            print(f"Handling leap year day {day}, using day 365 instead")
            day = 365
            
        # Get climatology for this day 
        clim_day = climatology['t_2m'].sel(dayofyear=day)
        
        # Interpolate to forecast grid
        clim_day_interp = clim_day.interp(
            latitude=daily_data.latitude,
            longitude=daily_data.longitude,
            method='linear'
        )
        
        # Store in the array
        clim_interp[i, :, :] = clim_day_interp.values
        
        # For first few days, print sample values
        if i < 3:
            lat_idx = len(daily_data.latitude) // 2
            lon_idx = len(daily_data.longitude) // 2
            sample_lat = daily_data.latitude.values[lat_idx]
            sample_lon = daily_data.longitude.values[lon_idx]
            
            forecast_val = daily_data['t_mean_2m_24h'].isel(time=i, latitude=lat_idx, longitude=lon_idx).values
            clim_val = clim_day_interp.isel(latitude=lat_idx, longitude=lon_idx).values
            anomaly_val = forecast_val - clim_val
            
            print(f"\nSample point at lat={sample_lat}, lon={sample_lon}, time index {i}:")
            print(f"  Forecast value: {forecast_val:.2f} deg C")
            print(f"  Climatology value: {clim_val:.2f} deg C")
            print(f"  Anomaly: {anomaly_val:.2f} deg C")
    
    # Convert back to xarray with proper dimensions
    climatology_interp = xr.DataArray(
        clim_interp,
        dims=('time', 'latitude', 'longitude'),
        coords={
            'time': daily_data.time,
            'latitude': daily_data.latitude,
            'longitude': daily_data.longitude
        }
    )
    
    print_stats(climatology_interp, "Interpolated Climatology")
    
    # Calculate anomalies
    anomalies = daily_data['t_mean_2m_24h'] - climatology_interp
    print_stats(anomalies, "Temperature Anomalies")
    
    # Create histograms to visualize distributions
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.hist(daily_data['t_mean_2m_24h'].values.flatten(), bins=50)
    plt.title('Forecast Temperature Distribution')
    plt.xlabel('Temperature (deg C)')
    
    plt.subplot(1, 3, 2)
    plt.hist(climatology_interp.values.flatten(), bins=50)
    plt.title('Climatology Temperature Distribution')
    plt.xlabel('Temperature (deg C)')
    
    plt.subplot(1, 3, 3)
    plt.hist(anomalies.values.flatten(), bins=50)
    plt.title('Anomaly Distribution')
    plt.xlabel('Temperature Anomaly (deg C)')
    
    plt.tight_layout()
    plt.savefig('temperature_distributions.png')
    plt.close()
    
    # Save anomalies
    anomalies_ds = xr.Dataset(
        {
            't_mean_2m_24h_anom': anomalies
        },
        coords={
            'time': daily_data['time'],
            'latitude': daily_data['latitude'],
            'longitude': daily_data['longitude']
        }
    )
    
    anomalies_ds.to_netcdf(anomalies_file)
    print(f"Created new anomalies file: {anomalies_file}")
else:
    print(f"Using existing anomalies file: {anomalies_file}")
    anomalies_ds = xr.open_dataset(anomalies_file)
    print_stats(anomalies_ds['t_mean_2m_24h_anom'], "Loaded Anomalies")

# Part 2: Calculate and Save Monthly Means
if not os.path.exists(monthly_means_file) or True:  # Force recalculation
    # Process time format
    ds = xr.open_dataset(anomalies_file)
    ds['time'] = pd.to_datetime(ds['time'].values)
    
    # Calculate monthly means
    anomalies = ds['t_mean_2m_24h_anom']
    monthly_means = anomalies.resample(time='M').mean()
    
    print_stats(monthly_means, "Monthly Mean Anomalies")
    
    # FIXED: Select months of interest properly
    # If you want specific months, uncomment and modify this:
    # months_of_interest = monthly_means.sel(time=monthly_means['time.month'].isin([4,5,6,7,8]))
    # Otherwise, use all months:
    months_of_interest = monthly_means
    
    print("\nMonthly means selected for months:")
    for i, time in enumerate(months_of_interest['time'].values):
        month_data = months_of_interest.sel(time=time)
        print(f"Month: {pd.Timestamp(time).month_name()}, Min: {month_data.min().values:.2f} deg C, Max: {month_data.max().values:.2f} deg C, Mean: {month_data.mean().values:.2f} deg C")
    
    # FIXED: Save as a proper dataset
    monthly_means_ds = xr.Dataset({
        't_mean_2m_24h_anom': months_of_interest
    })
    monthly_means_ds.to_netcdf(monthly_means_file)
    print(f"Created new monthly means file: {monthly_means_file}")
else:
    print(f"Using existing monthly means file: {monthly_means_file}")
    monthly_means_ds = xr.open_dataset(monthly_means_file)
    
    print("\nMonthly means from existing file:")
    for i, time in enumerate(monthly_means_ds['t_mean_2m_24h_anom']['time'].values):
        month_data = monthly_means_ds['t_mean_2m_24h_anom'].sel(time=time)
        print(f"Month: {pd.Timestamp(time).month_name()}, Min: {month_data.min().values:.2f} deg C, Max: {month_data.max().values:.2f} deg C, Mean: {month_data.mean().values:.2f} deg C")

# Part 3: Generate Individual Month Plots
print("\nGenerating individual month plots...")

# Load the monthly means
monthly_means_ds = xr.open_dataset(monthly_means_file)
anomalies = monthly_means_ds['t_mean_2m_24h_anom']

# Ensure time is datetime64
time_index = pd.to_datetime(anomalies['time'].values)

# Build month labels directly from time coordinate
month_labels = [t.strftime('%B') for t in time_index]      # Full month name
year_labels  = [t.strftime('%Y') for t in time_index]      # Year as string


projection = ccrs.PlateCarree(central_longitude=0)

# Generate individual month plots
for i, time_val in enumerate(anomalies['time']):
    month_name = month_labels[i]
    year_str   = year_labels[i]

    plot_file = rf'P:\QFA\TonyWeather\marta\LPG_REPORT\PLOTS\T2M_anomaly_{month_name}.png'
    
    if not os.path.exists(plot_file) or True:  # Force regeneration
        print(f"Creating plot for {month_name} {year_str}...")
        data = anomalies.sel(time=time_val)

        print(f"Data range for {month_name}: Min={data.min().values:.2f} deg C, "
              f"Max={data.max().values:.2f} deg C, Mean={data.mean().values:.2f} deg C")

        plt.figure(figsize=(15, 10))
        ax = plt.axes(projection=projection)
        ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())
        ax.coastlines()

        # Levels for anomalies
        levels = np.linspace(-10, 10, 13)

        contour = ax.contourf(
            data['longitude'], data['latitude'], data,
            cmap='RdBu_r', levels=levels, extend='both',
            transform=ccrs.PlateCarree()
        )

        ax.set_xticks(np.arange(-180, 181, 30), crs=ccrs.PlateCarree())
        ax.set_yticks(np.arange(0, 91, 15), crs=ccrs.PlateCarree())

        cbar = plt.colorbar(
            contour, ax=ax, orientation='horizontal',
            pad=0.08, fraction=0.046, label='T2m Anomaly (deg C)'
        )

        plt.title(f'Temperature anomaly at 2 m - {month_name} {year_str}')
        plt.tight_layout()
        plt.savefig(plot_file, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.close()
        print(f"Saved plot: {plot_file}")
    else:
        print(f"Plot already exists for {month_name}")

# Part 4: Generate Alternative Difference Plots
print("\nGenerating alternative difference plots (monthly anomaly vs. global mean)...")

# Reuse anomalies; compute mean over time only
global_mean_anomaly = anomalies.mean(dim=['time'])
print(f"Global mean anomaly: Min={global_mean_anomaly.min().values:.2f} deg C, "
      f"Max={global_mean_anomaly.max().values:.2f} deg C, "
      f"Mean={global_mean_anomaly.mean().values:.2f} deg C")

# Generate difference plots (each month vs global mean)
for i, time_val in enumerate(anomalies['time']):
    month_name = month_labels[i]
    year_str   = year_labels[i]

    diff_plot_file = rf'P:\QFA\TonyWeather\marta\LPG_REPORT\PLOTS\T2M_diff_from_mean_{month_name}.png'
    
    if not os.path.exists(diff_plot_file) or True:  # Force regeneration
        print(f"Creating difference plot for {month_name} {year_str}...")

        month_data = anomalies.sel(time=time_val)
        diff = month_data - global_mean_anomaly

        print(f"Difference from mean for {month_name}: Min={diff.min().values:.2f} deg C, "
              f"Max={diff.max().values:.2f} deg C, Mean={diff.mean().values:.2f} deg C")

        fig = plt.figure(figsize=(12, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.coastlines()

        levels = np.linspace(-3, 3, 13)

        diff_plot = diff.plot(
            ax=ax, transform=ccrs.PlateCarree(),
            cmap='RdBu_r', levels=levels, add_colorbar=False
        )

        cbar = plt.colorbar(
            diff_plot, ax=ax, orientation='horizontal',
            pad=0.08, fraction=0.046,
            label='Departure from Mean Anomaly (deg C)'
        )

        ax.set_xticks(np.arange(-180, 181, 30), crs=ccrs.PlateCarree())
        ax.set_yticks(np.arange(0, 91, 15), crs=ccrs.PlateCarree())
        ax.set_ylabel('')
        ax.set_xlabel('')

        ax.set_title(f'Difference from Mean Anomaly for {month_name} {year_str}', fontsize=14)

        plt.savefig(diff_plot_file, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.close()
        print(f"Saved difference plot: {diff_plot_file}")
    else:
        print(f"Difference plot already exists for {month_name}")

print("Script completed successfully!")


Loaded daily forecast data

Forecast data variables: ['crs_wgs84', 't_mean_2m_24h']
Forecast data dimensions: FrozenMappingWarningOnValuesAccess({'time': 151, 'latitude': 90, 'longitude': 360})
Forecast data time range: 2026-05-28T09:02:50.000000000 to 2026-10-25T09:02:50.000000000

--- Forecast Temperature Statistics ---
Shape: (151, 90, 360)
Data type: float64
Min: 230.8905792236328
Max: 318.7586669921875
Mean: 288.35611658687185
Median: 290.7212829589844
Has NaN: False
NaN count: 0
------------------------

Converting forecast data from Kelvin to Celsius...

--- Forecast Temperature (after K->C conversion) Statistics ---
Shape: (151, 90, 360)
Data type: float64
Min: -42.259420776367165
Max: 45.60866699218752
Mean: 15.206116586872032
Median: 17.571282958984398
Has NaN: False
NaN count: 0
------------------------

Loading climatology from: P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_1993_2023.nc
Loaded climatology data

Climatology data variables: ['t_2m']
Climatology data dime

Loaded daily forecast data

Forecast data variables: ['crs_wgs84', 't_mean_2m_24h']
Forecast data dimensions: FrozenMappingWarningOnValuesAccess({'time': 151, 'latitude': 90, 'longitude': 360})
Forecast data time range: 2026-05-28T09:02:50.000000000 to 2026-10-25T09:02:50.000000000

--- Forecast Temperature Statistics ---
Shape: (151, 90, 360)
Data type: float64
Min: 230.8905792236328
Max: 318.7586669921875
Mean: 288.35611658687185
Median: 290.7212829589844
Has NaN: False
NaN count: 0
------------------------

Converting forecast data from Kelvin to Celsius...

--- Forecast Temperature (after K->C conversion) Statistics ---
Shape: (151, 90, 360)
Data type: float64
Min: -42.259420776367165
Max: 45.60866699218752
Mean: 15.206116586872032
Median: 17.571282958984398
Has NaN: False
NaN count: 0
------------------------

Loading climatology from: P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_1993_2023.nc
Loaded climatology data

Climatology data variables: ['t_2m']
Climatology data dime

c:\Users\LORECOMI\AppData\Local\miniforge3\envs\snow_obs\Lib\site-packages\xarray\groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


Loaded daily forecast data

Forecast data variables: ['crs_wgs84', 't_mean_2m_24h']
Forecast data dimensions: FrozenMappingWarningOnValuesAccess({'time': 151, 'latitude': 90, 'longitude': 360})
Forecast data time range: 2026-05-28T09:02:50.000000000 to 2026-10-25T09:02:50.000000000

--- Forecast Temperature Statistics ---
Shape: (151, 90, 360)
Data type: float64
Min: 230.8905792236328
Max: 318.7586669921875
Mean: 288.35611658687185
Median: 290.7212829589844
Has NaN: False
NaN count: 0
------------------------

Converting forecast data from Kelvin to Celsius...

--- Forecast Temperature (after K->C conversion) Statistics ---
Shape: (151, 90, 360)
Data type: float64
Min: -42.259420776367165
Max: 45.60866699218752
Mean: 15.206116586872032
Median: 17.571282958984398
Has NaN: False
NaN count: 0
------------------------

Loading climatology from: P:\QFA\TonyWeather\marta\LPG_REPORT\t2_climatology_1993_2023.nc
Loaded climatology data

Climatology data variables: ['t_2m']
Climatology data dime

c:\Users\LORECOMI\AppData\Local\miniforge3\envs\snow_obs\Lib\site-packages\xarray\groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(



--- Monthly Mean Anomalies Statistics ---
Shape: (6, 90, 360)
Data type: float64
Min: -15.09404692619078
Max: 15.152376992248474
Mean: 0.9369574696684445
Median: 0.6821780517451105
Has NaN: False
NaN count: 0
------------------------


Monthly means selected for months:
Month: May, Min: -10.17 deg C, Max: 13.17 deg C, Mean: 1.28 deg C
Month: June, Min: -10.37 deg C, Max: 13.06 deg C, Mean: 0.92 deg C
Month: July, Min: -11.17 deg C, Max: 15.15 deg C, Mean: 0.88 deg C
Month: August, Min: -11.60 deg C, Max: 13.95 deg C, Mean: 0.69 deg C
Month: September, Min: -15.09 deg C, Max: 11.69 deg C, Mean: 0.82 deg C
Month: October, Min: -11.29 deg C, Max: 13.53 deg C, Mean: 1.03 deg C
Created new monthly means file: P:\QFA\TonyWeather\marta\LPG_REPORT\out_files\t_mean_2m_24h_anom_monthly_means_2026-05-28.nc

Generating individual month plots...
Creating plot for May 2026...
Data range for May: Min=-10.17 deg C, Max=13.17 deg C, Mean=1.28 deg C
Saved plot: P:\QFA\TonyWeather\marta\LPG_REPORT\PLOTS

In [30]:
import os
# Load MDS_SECRET from a .env file or set it manually if not already in the kernel environment
if not os.getenv("MDS_SECRET"):
    # Option 1: Load from dotenv file (if you have one)
    from dotenv import load_dotenv
    load_dotenv(r"P:\QFA\TonyWeather\.env")  # adjust path to your .env file
    
    # Option 2: If no .env file, uncomment and paste your secret here (DO NOT commit this):
    # os.environ["MDS_SECRET"] = "your-secret-here"

print("MDS_SECRET set:", "Yes" if os.getenv("MDS_SECRET") else "NO — please set it!")

MDS_SECRET set: Yes


In [39]:
# -*- coding: utf-8 -*-
"""
Multi-Region Weather Analysis Script using NetCDF Climatology (2024)
"""

import os
import numpy as np
import pandas as pd
import xarray as xr
from datetime import date, datetime, timedelta
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
from msal import ConfidentialClientApplication
import warnings

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

DSL_BASE_URL = "https://fs_gateway.prod.axponet.ch/dsl"
DSL_TENANT_ID = os.getenv("DSL_TENANT_ID", "8619c67c-945a-48ae-8e77-35b1b71c9b98")
DSL_CLIENT_ID = os.getenv("DSL_CLIENT_ID", "5a838d41-4163-42ef-8479-48fe7db6d547")
DSL_SCOPE = os.getenv("DSL_SCOPE", "api://e504750d-ac05-4807-a2e7-8fcd77b247aa/.default")
DSL_SECRET_ENV_NAME = os.getenv("DSL_SECRET_ENV_NAME", "MDS_SECRET")


def get_dsl_headers(secret_env_name=DSL_SECRET_ENV_NAME, client_id=DSL_CLIENT_ID, tenant_id=DSL_TENANT_ID, scope=DSL_SCOPE):
    client_secret = os.getenv(secret_env_name)
    if not client_secret:
        raise RuntimeError(
            f"Missing environment variable '{secret_env_name}'. Set it before calling the DSL API."
        )

    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = ConfidentialClientApplication(
        client_id=client_id,
        authority=authority,
        client_credential=client_secret,
    )
    token_result = app.acquire_token_for_client(scopes=[scope])

    access_token = token_result.get("access_token")
    if not access_token:
        error_message = token_result.get("error_description") or token_result.get("error") or "Unknown error"
        raise RuntimeError(f"Unable to fetch DSL OAuth token: {error_message}")

    return {
        "Authorization": f"Bearer {access_token}",
        "User-Agent": os.environ.get("USER") or os.environ.get("USERNAME") or "axpo-user",
        "Content-Type": "application/json",
        "Accept-Encoding": "gzip, deflate",
    }


# ------------------------------------------------------------
# 1. API WRAPPER
# ------------------------------------------------------------
def create_meteomatics_api(user=None, timeout=9000, secret_env_name=DSL_SECRET_ENV_NAME):
    base_url = f"{DSL_BASE_URL}/v3/meteo"

    regions = {
        "east_asia": "46.5,90_20,146.5:0.5,0.5",
        "europe": "72.5,-13_36,36.5:0.5,0.5",
        "us": "55.5,-130_28.5,-70:0.5,0.5"
    }

    def make_request(payload, output_format):
        # Get fresh headers for each request to avoid token expiry
        headers = get_dsl_headers(secret_env_name=secret_env_name)
        print(f"    → Sending request: source={payload.get('source')}, location={payload.get('location')[:20]}...")
        import time as _time
        t0 = _time.time()
        response = requests.post(base_url, json=payload, headers=headers, verify=False, timeout=timeout)
        elapsed = _time.time() - t0
        print(f"    ← Response: status={response.status_code}, time={elapsed:.1f}s, size={len(response.content)/1024:.0f}KB")
        if response.status_code >= 400:
            raise RuntimeError(
                f"DSL request failed ({response.status_code})\n"
                f"Payload: {payload}\n"
                f"Response: {response.text[:500]}"
            )
        return response.json() if output_format == 'json' else response.content

    def get_data(start_date, stop_date, region, granularity, parameters, output_format, model, ens_select="mean"):
        coords = regions[region.lower()]
        model_name = model
        if "&" in model_name:
            model_name = model_name.split("&", 1)[0]
        payload = {
            "parameters": parameters,
            "location": coords,
            "source": model_name,
            "datetime": f"{start_date}T00:00:00Z--{stop_date}T00:00:00Z:{granularity}",
            "outputFormat": output_format,
        }
        if ens_select:
            payload["source"] = f"{model_name}&ens_select={ens_select}"
        return make_request(payload, output_format)

    return {'get_data': get_data, 'regions': regions}


# ------------------------------------------------------------
# 2. LOAD CLIMATOLOGY (2024) FROM NETCDF
# ------------------------------------------------------------
def load_climatology_from_netcdf(nc_path, region_coords):
    ds = xr.open_dataset(nc_path)

    first, second = region_coords.split("_")
    lat_max, lon_min = map(float, first.split(","))
    lat_min, lon_max = map(float, second.split(":")[0].split(","))

    clim = ds.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max)
    )

    return clim


def calculate_deviations(forecast_data, climatology):
    import time as _time
    t0 = _time.time()

    clim_da = climatology["t_mean_2m_24h"].load()  # Load into memory for fast access
    clim_lats = clim_da.latitude.values
    clim_lons = clim_da.longitude.values
    # Pre-convert to numpy for fastest access
    clim_values = clim_da.values  # shape: (dayofyear, lat, lon)
    doy_values = clim_da.dayofyear.values

    print(f"    Climatology loaded in {_time.time()-t0:.1f}s, shape={clim_values.shape}")
    print(f"    Forecast points: {len(forecast_data)}")
    t0 = _time.time()

    deviations = []
    for point in forecast_data:
        lat = point["lat"]
        lon = point["lon"]

        lat_idx = np.argmin(np.abs(clim_lats - lat))
        lon_idx = np.argmin(np.abs(clim_lons - lon))

        for d in point["dates"]:
            forecast_date = pd.to_datetime(d["date"])
            doy = forecast_date.timetuple().tm_yday

            # Direct numpy array indexing
            doy_idx = np.searchsorted(doy_values, doy)
            if doy_idx >= len(doy_values) or doy_values[doy_idx] != doy:
                continue

            clim_value = clim_values[doy_idx, lat_idx, lon_idx]
            if np.isnan(clim_value):
                continue

            deviation = d["value"] - clim_value
            deviations.append((lat, lon, d["date"], deviation))

    dtype = [('lat','float'), ('lon','float'),
             ('date','U25'), ('temperature_deviation','float')]

    print(f"    Deviations computed in {_time.time()-t0:.1f}s, {len(deviations)} points")
    return np.array(deviations, dtype=dtype)



# ------------------------------------------------------------
# 4. SAVE CSV
# ------------------------------------------------------------
def save_deviations_to_csv(deviation_array, filename):
    np.savetxt(filename, deviation_array, delimiter=',', fmt='%s',
               header='lat,lon,date,temperature_deviation', comments='')


# ------------------------------------------------------------
# 5. GROUP DEVIATIONS INTO PERIODS
# ------------------------------------------------------------
def load_and_process_deviation_data(csv_path):
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    daily_avg = df.groupby('date')['temperature_deviation'].mean().reset_index()
    daily_avg.columns = ['date', 'avg_deviation']
    return daily_avg


# ------------------------------------------------------------
# 6. PLOT MAP
# ------------------------------------------------------------
def plot_temperature_deviation_map(data, lat_grid, lon_grid, start_date, period_idx, save_dir, region):
    fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={'projection': ccrs.PlateCarree()})

    ax.set_extent([lon_grid.min(), lon_grid.max(), lat_grid.min(), lat_grid.max()],
                  crs=ccrs.PlateCarree())

    levels = np.arange(-6, 7, 1)
    cs = ax.contourf(lon_grid, lat_grid, data, levels=levels,
                     cmap='RdBu_r', extend='both')

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)

    cbar = plt.colorbar(cs, ax=ax)
    cbar.set_label("Temperature Deviation (deg C)")

    start_dt = pd.to_datetime(start_date)
    end_dt = start_dt + timedelta(days=5)

    title = (
        f"{region.upper()} - Temperature Deviation\n"
        f"{start_dt.strftime('%Y-%m-%d')} to {end_dt.strftime('%Y-%m-%d')}"
    )
    plt.title(title, fontsize=13)

    filename = os.path.join(
        save_dir,
        f"temperature_deviation_{region}_period_{period_idx}.png"
    )

    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

# ------------------------------------------------------------
# 7. MAIN
# ------------------------------------------------------------
def main():
    api_user = 'IGNAOLSZ'
    nc_path = r"P:\QFA\TonyWeather\CLIMATOLOGIES\temperature\ERA5\data_0.5x0.5\detrended_climatology_T2.nc"

    regions_to_process = ["east_asia", "us", "europe"]

    api = create_meteomatics_api(api_user)
    save_dir = r"P:\QFA\TonyWeather\marta\LPG_REPORT\maps"
    os.makedirs(save_dir, exist_ok=True)

    for region in regions_to_process:
        print(f"\n=== Processing region: {region} ===")

        start_date = date.today().strftime("%Y-%m-%d")
        stop_date = (date.today() + timedelta(days=14)).strftime("%Y-%m-%d")

        forecast_json = api["get_data"](
            start_date=start_date,
            stop_date=stop_date,
            region=region,
            granularity="P1D",
            parameters="t_mean_2m_24h:C",
            output_format="json",
            model="ecmwf-ens"
        )

        forecast_data = forecast_json["data"][0]["coordinates"]

        region_coords = api["regions"][region]
        climatology_2024 = load_climatology_from_netcdf(nc_path, region_coords)

        deviation_array = calculate_deviations(forecast_data, climatology_2024)

        csv_dev = f"deviations_{region}.csv"
        save_deviations_to_csv(deviation_array, csv_dev)

        combined_data = load_and_process_deviation_data(csv_dev)
        combined_data.to_csv(f"average_temperature_deviations_{region}.csv", index=False)

        # Use raw deviations for plotting (has lat, lon, date, temperature_deviation)
        raw_dev = pd.read_csv(csv_dev)
        raw_dev['date'] = pd.to_datetime(raw_dev['date'])

        first, second = region_coords.split("_")
        lat_max, lon_min = map(float, first.split(","))
        lat_min, lon_max = map(float, second.split(":")[0].split(","))

        lat_grid = np.linspace(lat_min, lat_max, 200)
        lon_grid = np.linspace(lon_min, lon_max, 200)
        lon_grid, lat_grid = np.meshgrid(lon_grid, lat_grid)

        period_idx = 1
        unique_dates = sorted(raw_dev["date"].dt.date.unique())
        # Group into 5-day periods: days 1-5, 6-10, 11-15
        for i in range(0, len(unique_dates), 5):
            period_dates = unique_dates[i:i+5]
            if not period_dates:
                continue

            period_data = raw_dev[raw_dev["date"].dt.date.isin(period_dates)]
            # Average deviation per grid point over the 5-day period
            avg_period = period_data.groupby(["lat", "lon"])["temperature_deviation"].mean().reset_index()

            points = np.column_stack([avg_period["lat"], avg_period["lon"]])
            values = avg_period["temperature_deviation"]

            grid_data = griddata(points, values, (lat_grid, lon_grid), method='cubic', fill_value=np.nan)

            if not np.isnan(grid_data).all():
                plot_temperature_deviation_map(
                    grid_data, lat_grid, lon_grid,
                    start_date=str(period_dates[0]),
                    period_idx=period_idx,
                    save_dir=save_dir,
                    region=region
                )
                period_idx += 1


# ------------------------------------------------------------
# EXECUTION
# ------------------------------------------------------------
if __name__ == "__main__":
    main()



=== Processing region: east_asia ===
    → Sending request: source=ecmwf-ens&ens_select=mean, location=46.5,90_20,146.5:0.5...
    ← Response: status=200, time=6.7s, size=4247KB
    Climatology loaded in 6.2s, shape=(365, 54, 114)
    Forecast points: 6156
    Deviations computed in 18.6s, 92340 points

=== Processing region: us ===
    → Sending request: source=ecmwf-ens&ens_select=mean, location=55.5,-130_28.5,-70:0...
    ← Response: status=200, time=7.1s, size=4596KB
    Climatology loaded in 6.0s, shape=(365, 55, 121)
    Forecast points: 6655
    Deviations computed in 20.0s, 99825 points

=== Processing region: europe ===
    → Sending request: source=ecmwf-ens&ens_select=mean, location=72.5,-13_36,36.5:0.5...
    ← Response: status=200, time=8.4s, size=5083KB
    Climatology loaded in 8.6s, shape=(365, 74, 100)
    Forecast points: 7400
    Deviations computed in 22.1s, 111000 points


In [32]:
def select_precip_var(data_vars):
    """Select precipitation variable from a dataset."""
    precip_vars = ['precip_24h:mm', 'precip_24h', 'pr', 'precipitation', 'total_precipitation', 'precip']
    for var in precip_vars:
        if var in data_vars:
            return var
    raise ValueError(f"No precipitation variable found. Available: {list(data_vars)}")


def dsl_authentication_for_precip(app_id='5a838d41-4163-42ef-8479-48fe7db6d547', env_name='MDS_SECRET'):
    """Authenticate with DSL API using the same OAuth flow as the working downloader."""
    axpo_azure_tenant_id = '8619c67c-945a-48ae-8e77-35b1b71c9b98'
    app_authority_url = 'https://login.microsoftonline.com/' + axpo_azure_tenant_id
    dsl_api_app_read_scope = 'api://e504750d-ac05-4807-a2e7-8fcd77b247aa/.default'

    client_secret = os.getenv(env_name)
    if not client_secret:
        raise RuntimeError(f"Environment variable {env_name} not found. Please set it and restart the kernel.")

    authentication_object = ConfidentialClientApplication(
        client_id=app_id,
        authority=app_authority_url,
        client_credential=client_secret,
    )
    result = authentication_object.acquire_token_for_client(scopes=[dsl_api_app_read_scope])

    if 'access_token' not in result:
        error_msg = result.get('error_description', 'Unknown error')
        raise RuntimeError(f'Unable to fetch token: {error_msg}')

    return {
        'Authorization': f"Bearer {result['access_token']}",
        'User-Agent': os.environ.get('USER', os.environ.get('USERNAME', 'axpo-user')),
        'Content-Type': 'application/json',
        'Accept-Encoding': 'gzip, deflate',
    }


def build_precip_dataset_from_json(precip_json, var_name='precip_24h:mm'):
    """Convert DSL JSON coordinates into gridded xarray dataset: time x lat x lon."""
    coords = precip_json['data'][0]['coordinates']
    if not coords:
        return None

    lats = np.array(sorted({float(c['lat']) for c in coords}))
    lons = np.array(sorted({float(c['lon']) for c in coords}))
    times = pd.to_datetime([d['date'] for d in coords[0]['dates']])

    lat_idx = {v: i for i, v in enumerate(lats)}
    lon_idx = {v: i for i, v in enumerate(lons)}
    grid = np.full((len(times), len(lats), len(lons)), np.nan, dtype=float)

    for c in coords:
        i = lat_idx[float(c['lat'])]
        j = lon_idx[float(c['lon'])]
        for t_idx, d in enumerate(c['dates']):
            if t_idx < len(times):
                grid[t_idx, i, j] = d.get('value', np.nan)

    return xr.Dataset({var_name: (['time', 'lat', 'lon'], grid)}, coords={'time': times, 'lat': lats, 'lon': lons})


def fetch_precip_data_for_region(forecast_start, forecast_end, region_name):
    """Fetch region gridded precipitation as xarray using direct OAuth-authenticated DSL requests."""
    region_coords = {
        'east_asia': '46.5,90_20,146.5:0.5,0.5',
        'europe': '72.5,-13_36,36.5:0.5,0.5',
        'us': '55.5,-130_28.5,-70:0.5,0.5',
    }
    if region_name not in region_coords:
        raise RuntimeError(f'Unsupported region: {region_name}')

    headers = dsl_authentication_for_precip()
    payload = {
        'parameters': 'precip_24h:mm',
        'location': region_coords[region_name],
        'source': 'ecmwf-ens',
        'datetime': f"{forecast_start}T00:00:00Z--{forecast_end}T00:00:00Z:P1D",
        'outputFormat': 'json',
    }

    response = requests.post(
        'https://fs_gateway.prod.axponet.ch/dsl/v3/meteo',
        json=payload,
        headers=headers,
        verify=False,
        timeout=300,
    )
    if response.status_code >= 400:
        raise RuntimeError(
            f"DSL request failed ({response.status_code})\n"
            f"Payload: {payload}\n"
            f"Response: {response.text[:500]}"
        )

    return build_precip_dataset_from_json(response.json())


def load_precip_climatology():
    """Load precipitation climatology from known kernel paths or standard fallbacks."""
    candidate_paths = []
    if 'precip_clim_path' in globals() and precip_clim_path:
        candidate_paths.append(precip_clim_path)
    if 'precip_clim_candidates' in globals() and precip_clim_candidates:
        candidate_paths.extend(list(precip_clim_candidates))

    candidate_paths.extend([
        r'P:\QFA\TonyWeather\CLIMATOLOGIES\precipitation\ERA5\data_1x1\climatology_precip.nc',
        r'P:\QFA\TonyWeather\CLIMATOLOGIES\precipitation\ERA5\data_0.5x0.5\climatology_precip.nc',
        'east_asia_precip_climatology.nc',
    ])

    seen = set()
    for path in candidate_paths:
        if not path or path in seen:
            continue
        seen.add(path)
        if os.path.exists(path):
            ds = xr.open_dataset(path)
            rename_map = {}
            if 'latitude' in ds.coords:
                rename_map['latitude'] = 'lat'
            if 'longitude' in ds.coords:
                rename_map['longitude'] = 'lon'
            if rename_map:
                ds = ds.rename(rename_map)
            print(f'Loaded precipitation climatology: {path}')
            return ds

    raise FileNotFoundError(f'No precipitation climatology file found. Checked: {candidate_paths}')


def compute_precip_deviation_dataset(precip_data, climatology_data):
    """Interpolate daily climatology to forecast grid and compute deviations."""
    forecast_var = select_precip_var(precip_data.data_vars)
    clim_var = select_precip_var(climatology_data.data_vars)

    deviation_values = np.full_like(precip_data[forecast_var].values, np.nan, dtype=float)

    for time_idx, time_value in enumerate(pd.to_datetime(precip_data.time.values)):
        day_of_year = min(time_value.dayofyear, 365)
        clim_slice = climatology_data[clim_var].sel(dayofyear=day_of_year)
        clim_interp = clim_slice.interp(lat=precip_data.lat.values, lon=precip_data.lon.values)
        deviation_values[time_idx] = precip_data[forecast_var].isel(time=time_idx).values - clim_interp.values

    return xr.Dataset(
        {'precip_deviation': (['time', 'lat', 'lon'], deviation_values)},
        coords={'time': precip_data.time.values, 'lat': precip_data.lat.values, 'lon': precip_data.lon.values},
    )


def load_and_process_precip_deviation_data(forecast_start, forecast_end, region_name):
    """Load precip deviations, group into periods, and interpolate missing data."""
    try:
        precip_data = fetch_precip_data_for_region(forecast_start, forecast_end, region_name)
        climatology_data = load_precip_climatology()
        deviation_data = compute_precip_deviation_dataset(precip_data, climatology_data)
    except Exception as e:
        print(f"Error fetching precipitation: {e}")
        return None, None, None, None

    lat_grid, lon_grid = np.meshgrid(deviation_data.lat.values, deviation_data.lon.values, indexing='ij')
    periods_data = []
    period_bounds = [(0, 5), (5, 10), (10, 15)]
    # Also return the actual time values so we can use real dates in plot titles
    times = pd.to_datetime(deviation_data.time.values)

    for start_idx, end_idx in period_bounds:
        period_slice = deviation_data['precip_deviation'].isel(time=slice(start_idx, end_idx)).mean('time')
        period_data = interpolate_precip_grid(lon_grid, lat_grid, period_slice.values)
        periods_data.append(period_data)

    return periods_data, lat_grid, lon_grid, times


def interpolate_precip_grid(lon_grid, lat_grid, data):
    """Fill NaN values via multi-method fallback: cubic → linear → nearest."""
    from scipy.interpolate import griddata

    mask = ~np.isnan(data)
    if mask.sum() == 0:
        return data

    points = np.column_stack([lon_grid[mask], lat_grid[mask]])
    values = data[mask]
    xi = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])

    try:
        grid_cubic = griddata(points, values, xi, method='cubic')
        grid_cubic = grid_cubic.reshape(data.shape)
    except Exception:
        grid_cubic = np.full_like(data, np.nan)

    if np.isnan(grid_cubic).sum() > 0:
        try:
            grid_linear = griddata(points, values, xi, method='linear').reshape(data.shape)
            grid_cubic[np.isnan(grid_cubic)] = grid_linear[np.isnan(grid_cubic)]
        except Exception:
            pass

    if np.isnan(grid_cubic).sum() > 0:
        try:
            grid_nearest = griddata(points, values, xi, method='nearest').reshape(data.shape)
            grid_cubic[np.isnan(grid_cubic)] = grid_nearest[np.isnan(grid_cubic)]
        except Exception:
            pass

    return grid_cubic


YANGTZE_BOUNDARY_CACHE = None


def get_yangtze_basin_boundary():
    """Load precise Yangtze basin boundary from local GeoJSON or HydroBASINS."""
    global YANGTZE_BOUNDARY_CACHE
    if YANGTZE_BOUNDARY_CACHE is not None:
        return YANGTZE_BOUNDARY_CACHE

    try:
        import geopandas as gpd
        from shapely.geometry import LineString
        import tempfile
        import warnings
        import requests

        warnings.filterwarnings('ignore', message='Unverified HTTPS request')
        local_geojson = r'P:\QFA\TonyWeather\marta\LPG_REPORT\yangtze_basin_precise.geojson'
        if os.path.exists(local_geojson):
            YANGTZE_BOUNDARY_CACHE = gpd.read_file(local_geojson)
            print('Loaded Yangtze basin boundary from local GeoJSON cache.')
            return YANGTZE_BOUNDARY_CACHE

        url = 'https://data.hydrosheds.org/file/hydrobasins/standard/hybas_as_lev12_v1c.zip'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
        }
        with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp:
            tmp_zip = tmp.name

        response = requests.get(url, headers=headers, timeout=300, verify=False)
        response.raise_for_status()
        with open(tmp_zip, 'wb') as f:
            f.write(response.content)

        basins = gpd.read_file(tmp_zip)

        if basins.crs is None:
            basins = basins.set_crs('EPSG:4326')
        elif basins.crs.to_string() != 'EPSG:4326':
            basins = basins.to_crs('EPSG:4326')

        yangtze = basins[basins['MAIN_BAS'].astype(str) == '3140405440'] if 'MAIN_BAS' in basins.columns else basins.iloc[0:0].copy()

        if yangtze.empty:
            river_line = LineString([
                (91.0, 34.5), (95.0, 33.8), (99.0, 32.0), (103.0, 30.8),
                (107.0, 30.6), (111.0, 30.5), (114.5, 30.6), (118.5, 31.2), (121.7, 31.3)
            ])
            search_geom = gpd.GeoSeries([river_line], crs='EPSG:4326').buffer(0.6).iloc[0]
            touched = basins[basins.intersects(search_geom)]
            if 'MAIN_BAS' in touched.columns and not touched.empty:
                main_bas = touched['MAIN_BAS'].astype(str).value_counts().index[0]
                yangtze = basins[basins['MAIN_BAS'].astype(str) == main_bas]
            else:
                yangtze = touched.copy()

        if yangtze.empty:
            raise RuntimeError('Could not isolate Yangtze basin from HydroBASINS.')

        YANGTZE_BOUNDARY_CACHE = yangtze.dissolve(by=None)
        try:
            YANGTZE_BOUNDARY_CACHE.to_file(local_geojson, driver='GeoJSON')
        except Exception:
            pass
        try:
            os.remove(tmp_zip)
        except Exception:
            pass
        print(f"Loaded Yangtze basin boundary from HydroBASINS ({len(yangtze)} sub-basins).")
        return YANGTZE_BOUNDARY_CACHE

    except Exception as e:
        print(f"Warning: HydroBASINS boundary load failed; using fallback outline. Details: {e}")
        YANGTZE_BOUNDARY_CACHE = None
        return None


def add_yangtze_basin_outline(ax):
    """Add precise Yangtze basin boundary (HydroBASINS), with polyline fallback."""
    boundary = get_yangtze_basin_boundary()

    if boundary is not None and not boundary.empty:
        ax.add_geometries(
            boundary.geometry,
            crs=ccrs.PlateCarree(),
            facecolor='none',
            edgecolor='darkblue',
            linewidth=1.8,
            linestyle='-',
            alpha=0.9,
            zorder=4
        )
    else:
        yangtze_lon = [90.0, 93.0, 96.0, 99.0, 102.0, 105.0, 108.0, 111.0, 114.0, 117.0, 120.0, 122.0,
                       121.0, 119.0, 116.0, 113.0, 110.0, 107.0, 104.0, 101.0, 98.0, 95.0, 92.0, 90.0]
        yangtze_lat = [33.0, 33.0, 32.0, 31.5, 31.0, 31.0, 31.5, 32.0, 32.0, 31.5, 31.0, 30.0,
                       28.5, 27.5, 27.0, 27.0, 27.5, 28.0, 28.0, 27.5, 27.0, 27.5, 30.0, 33.0]
        ax.plot(yangtze_lon, yangtze_lat, color='darkblue', linewidth=1.6, linestyle='--', transform=ccrs.PlateCarree(), zorder=4, alpha=0.85)

    from matplotlib.lines import Line2D
    legend_line = Line2D([0], [0], color='darkblue', linewidth=1.8, linestyle='-')
    ax.legend([legend_line], ['Yangtze Basin Boundary'], loc='lower left', fontsize=8, frameon=True, fancybox=True, shadow=True)


# ---------------------------------------------------------------------
# Three Gorges Dam catchment helpers
# ---------------------------------------------------------------------
THREE_GORGES_SHP = r'P:\QFA\TonyWeather\marta\LPG_REPORT\shapefiles\three_gorges_catchment.shp'
THREE_GORGES_DAM_LON, THREE_GORGES_DAM_LAT = 111.0056, 30.8233
_THREE_GORGES_CACHE = None


def get_three_gorges_catchment():
    """Load and cache the Three Gorges Dam upstream catchment polygon (EPSG:4326)."""
    global _THREE_GORGES_CACHE
    if _THREE_GORGES_CACHE is not None:
        return _THREE_GORGES_CACHE
    import geopandas as gpd
    gdf = gpd.read_file(THREE_GORGES_SHP)
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    _THREE_GORGES_CACHE = gdf
    print(f'Loaded Three Gorges catchment ({len(gdf)} feature[s]) from {THREE_GORGES_SHP}')
    return gdf


def add_three_gorges_catchment_outline(ax):
    """Overlay the Three Gorges catchment boundary (no legend; caller sets the legend)."""
    try:
        gdf = get_three_gorges_catchment()
    except Exception as e:
        print(f'Warning: could not load Three Gorges catchment: {e}')
        return
    ax.add_geometries(
        gdf.geometry, crs=ccrs.PlateCarree(),
        facecolor='none', edgecolor='black', linewidth=1.8, linestyle='-', zorder=5
    )


def _three_gorges_grid_mask(lons, lats):
    """Boolean (lat, lon) mask of grid-cell centres that fall inside the catchment."""
    gdf = get_three_gorges_catchment()
    try:
        geom = gdf.geometry.union_all()
    except AttributeError:
        geom = gdf.geometry.unary_union
    lon2d, lat2d = np.meshgrid(np.asarray(lons), np.asarray(lats))
    try:
        from shapely.vectorized import contains
        mask = contains(geom, lon2d, lat2d)
    except Exception:
        from shapely.geometry import Point
        mask = np.array([[geom.contains(Point(float(x), float(y))) for x in lons] for y in lats])
    return mask


def compute_three_gorges_daily_precip_deviation(forecast_start, forecast_end):
    """Daily catchment-mean precip deviation vs climatology, plus the full grid for mapping."""
    precip_data = fetch_precip_data_for_region(forecast_start, forecast_end, 'east_asia')
    climatology_data = load_precip_climatology()
    deviation_data = compute_precip_deviation_dataset(precip_data, climatology_data)

    lons = deviation_data.lon.values
    lats = deviation_data.lat.values
    mask = _three_gorges_grid_mask(lons, lats)
    if mask.sum() == 0:
        print('WARNING: no precipitation grid cells fall inside the Three Gorges catchment.')

    da = deviation_data['precip_deviation']
    mask_da = xr.DataArray(mask, dims=('lat', 'lon'), coords={'lat': da.lat, 'lon': da.lon})
    catchment_series = da.where(mask_da).mean(dim=('lat', 'lon'), skipna=True)

    times = pd.to_datetime(deviation_data.time.values)
    series = pd.Series(catchment_series.values, index=times, name='precip_deviation_mm_day')
    return series, mask, deviation_data


# --- Last-5-years (raw ERA5) catchment-mean precip anomaly band ---
ERA5_PRECIP_FILE = r'P:\QFA\TonyWeather\CLIMATOLOGIES\precipitation\ERA5\data_1x1\C_precip_1979-2024.nc'
LAST5_YEARS = list(range(datetime.today().year - 4, datetime.today().year + 1))


def compute_three_gorges_5yr_anomaly_band(forecast_index, years=LAST5_YEARS,
                                          era5_path=ERA5_PRECIP_FILE):
    """For each forecast day, the last-5-years catchment-mean precipitation
    anomaly (raw ERA5 minus the SAME climatology used for the forecast), returned
    as mean / min / max across the chosen years on matching day-of-year."""
    gdf = get_three_gorges_catchment()
    minx, miny, maxx, maxy = gdf.total_bounds

    # raw ERA5 daily precip -> catchment-mean time series
    era5 = xr.open_dataset(era5_path)['precip_24h']
    if era5.latitude.values[0] > era5.latitude.values[-1]:
        era5 = era5.sortby('latitude')
    if era5.longitude.values[0] > era5.longitude.values[-1]:
        era5 = era5.sortby('longitude')
    era5 = era5.sel(latitude=slice(miny - 1, maxy + 1),
                    longitude=slice(minx - 1, maxx + 1))
    em = _three_gorges_grid_mask(era5.longitude.values, era5.latitude.values)
    em_da = xr.DataArray(em, dims=('latitude', 'longitude'),
                         coords={'latitude': era5.latitude, 'longitude': era5.longitude})
    era5_cm = era5.where(em_da).mean(('latitude', 'longitude'), skipna=True).to_pandas()
    era5_cm.index = pd.to_datetime(era5_cm.index).normalize()

    # same climatology -> catchment-mean per day-of-year
    clim = load_precip_climatology()
    cvar = select_precip_var(clim.data_vars)
    cm = _three_gorges_grid_mask(clim.lon.values, clim.lat.values)
    cm_da = xr.DataArray(cm, dims=('lat', 'lon'),
                         coords={'lat': clim.lat, 'lon': clim.lon})
    clim_cm = clim[cvar].where(cm_da).mean(('lat', 'lon'), skipna=True)
    clim_by_doy = {int(d): float(v) for d, v in zip(clim_cm.dayofyear.values, clim_cm.values)}

    rows = {'mean': [], 'min': [], 'max': []}
    idx = []
    for d in pd.to_datetime(forecast_index):
        cl = clim_by_doy.get(min(int(d.dayofyear), 365), np.nan)
        vals = []
        for y in years:
            try:
                v = era5_cm.loc[pd.Timestamp(year=y, month=d.month, day=d.day)]
            except KeyError:
                v = np.nan
            if pd.notna(v) and pd.notna(cl):
                vals.append(float(v) - cl)
        idx.append(d)
        rows['mean'].append(np.mean(vals) if vals else np.nan)
        rows['min'].append(np.min(vals) if vals else np.nan)
        rows['max'].append(np.max(vals) if vals else np.nan)
    return pd.DataFrame(rows, index=pd.DatetimeIndex(idx))


def plot_three_gorges_deviation_line(series, save_path, band=None):
    """Line plot of daily catchment-mean precipitation deviation from climatology."""
    mean_dev = float(np.nanmean(series.values))
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.axhline(0, color='grey', linewidth=1, linestyle='--')

    # last-5-years climatological envelope (same anomaly baseline)
    if band is not None and not band.dropna(how='all').empty:
        ax.fill_between(band.index, band['min'], band['max'],
                        color='steelblue', alpha=0.35, zorder=1,
                        label=f'Last 5 yrs ({LAST5_YEARS[0]}–{LAST5_YEARS[-1]}) min–max')
        ax.plot(band.index, band['mean'], color='steelblue', linewidth=2.0,
                linestyle='-.', zorder=2, label=f'Last 5 yrs ({LAST5_YEARS[0]}–{LAST5_YEARS[-1]}) mean')

    ax.plot(series.index, series.values, color='tab:blue', linewidth=2, marker='o',
            markersize=3, zorder=3, label='Forecast daily deviation')
    ax.fill_between(series.index, 0, series.values, where=(series.values >= 0),
                    color='tab:blue', alpha=0.25, interpolate=True)
    ax.fill_between(series.index, 0, series.values, where=(series.values < 0),
                    color='tab:red', alpha=0.25, interpolate=True)
    ax.axhline(mean_dev, color='black', linewidth=1.3, linestyle=':',
               label=f'Forecast-period mean = {mean_dev:+.2f} mm/day')
    ax.set_title('Three Gorges Catchment - Daily Precipitation Deviation from Climatology')
    ax.set_ylabel('Precipitation deviation (mm/day)')
    ax.set_xlabel('Date')
    ax.legend(loc='best', fontsize=9, frameon=True)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_three_gorges_catchment_map(deviation_data, save_path):
    """Zoomed map of mean forecast precip deviation over the catchment (outline + dam marker)."""
    field = deviation_data['precip_deviation'].mean('time')
    gdf = get_three_gorges_catchment()
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 2.0

    fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent([minx - pad, maxx + pad, miny - pad, maxy + pad], crs=ccrs.PlateCarree())
    levels = np.arange(-20, 22, 2)
    cs = ax.contourf(field.lon, field.lat, field.values, levels=levels,
                     cmap='BrBG', extend='both', transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    add_three_gorges_catchment_outline(ax)
    ax.plot(THREE_GORGES_DAM_LON, THREE_GORGES_DAM_LAT, marker='*', color='red',
            markersize=15, transform=ccrs.PlateCarree(), zorder=6)

    from matplotlib.lines import Line2D as _L2D
    ax.legend(
        [_L2D([0], [0], color='black', linewidth=1.8),
         _L2D([0], [0], marker='*', color='red', linestyle='None', markersize=12)],
        ['Catchment boundary', 'Three Gorges Dam'],
        loc='lower left', fontsize=8, frameon=True,
    )

    cbar = plt.colorbar(cs, ax=ax)
    cbar.set_label('Precipitation Deviation (mm/day)')
    ax.set_title('Three Gorges Catchment - Mean Forecast Precipitation Deviation')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_precipitation_deviation_map(data, lat_grid, lon_grid, period_start, period_end, period_idx, save_dir, region):
    fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={'projection': ccrs.PlateCarree()})

    ax.set_extent([lon_grid.min(), lon_grid.max(), lat_grid.min(), lat_grid.max()], crs=ccrs.PlateCarree())
    levels = np.arange(-20, 22, 2)
    cs = ax.contourf(lon_grid, lat_grid, data, levels=levels, cmap='BrBG', extend='both')

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)

    if region == 'east_asia':
        add_yangtze_basin_outline(ax)
        add_three_gorges_catchment_outline(ax)
        from matplotlib.lines import Line2D as _L2D
        ax.legend(
            [_L2D([0], [0], color='darkblue', linewidth=1.8),
             _L2D([0], [0], color='black', linewidth=1.8)],
            ['Yangtze Basin Boundary', 'Three Gorges Catchment'],
            loc='lower left', fontsize=8, frameon=True, fancybox=True, shadow=True,
        )

    cbar = plt.colorbar(cs, ax=ax)
    cbar.set_label('Precipitation Deviation (mm/day)')

    start_dt = pd.to_datetime(period_start)
    end_dt = pd.to_datetime(period_end)

    title = (
        f"{region.upper()} - Precipitation Deviation\n"
        f"{start_dt.strftime('%Y-%m-%d')} to {end_dt.strftime('%Y-%m-%d')}"
    )
    plt.title(title, fontsize=13)

    filename = os.path.join(save_dir, f'precipitation_deviation_{region}_period_{period_idx}.png')
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()


# Explicitly set save directory for precipitation maps
precip_save_dir = r'P:\QFA\TonyWeather\marta\LPG_REPORT\maps'
os.makedirs(precip_save_dir, exist_ok=True)

print('\n=== Generating East Asia Precipitation Deviation Maps ===')
precip_start_date = pd.Timestamp.today().strftime('%Y-%m-%d')
precip_end_date = (pd.Timestamp.today() + pd.Timedelta(days=15)).strftime('%Y-%m-%d')
periods_precip, lat_precip, lon_precip, precip_times = load_and_process_precip_deviation_data(
    precip_start_date,
    precip_end_date,
    'east_asia'
)

if periods_precip is not None:
    period_bounds = [(0, 5), (5, 10), (10, 15)]
    for idx, (period_data, (s_idx, e_idx)) in enumerate(zip(periods_precip, period_bounds), 1):
        # Use actual dates from the data for correct plot titles
        p_start = precip_times[s_idx]
        p_end   = precip_times[min(e_idx - 1, len(precip_times) - 1)]
        plot_precipitation_deviation_map(period_data, lat_precip, lon_precip, p_start, p_end, idx, precip_save_dir, 'east_asia')
        print(f'  Period {idx}: {p_start.strftime("%Y-%m-%d")} to {p_end.strftime("%Y-%m-%d")} -> precipitation_deviation_east_asia_period_{idx}.png')

    nan_coverage_pct = sum(np.isnan(p).sum() for p in periods_precip) / sum(p.size for p in periods_precip) * 100

    print(f"\nPrecipitation NaN coverage after interpolation: {nan_coverage_pct:.2f}%")
else:
    print('  Skipped: Unable to retrieve precipitation data')


# =====================================================================
# Three Gorges Dam catchment: outline map + catchment-mean daily
# precipitation deviation line (anomaly vs climatology)
# =====================================================================
print('\n=== Generating Three Gorges catchment precipitation deviation plots ===')
try:
    tg_series, tg_mask, tg_dev = compute_three_gorges_daily_precip_deviation(
        precip_start_date, precip_end_date
    )
    tg_line_png = os.path.join(precip_save_dir, 'three_gorges_precip_deviation_line.png')
    tg_map_png = os.path.join(precip_save_dir, 'three_gorges_catchment_precip_map.png')
    tg_band = compute_three_gorges_5yr_anomaly_band(tg_series.index)
    plot_three_gorges_deviation_line(tg_series, tg_line_png, band=tg_band)
    plot_three_gorges_catchment_map(tg_dev, tg_map_png)
    print(f'  5-yr band years used: {LAST5_YEARS} '
          f'({int(tg_band["mean"].notna().sum())}/{len(tg_band)} days covered)')
    print(f'  Catchment grid cells used: {int(tg_mask.sum())}')
    print(f'  Forecast-period mean deviation: {tg_series.mean():+.2f} mm/day')
    print(f'  Saved line plot:    {tg_line_png}')
    print(f'  Saved catchment map: {tg_map_png}')
except Exception as e:
    print(f'  Three Gorges catchment plots failed: {e}')



=== Generating East Asia Precipitation Deviation Maps ===
Loaded precipitation climatology: P:\QFA\TonyWeather\CLIMATOLOGIES\precipitation\ERA5\data_1x1\climatology_precip.nc
Loaded Yangtze basin boundary from local GeoJSON cache.
  Period 1: 2026-05-28 to 2026-06-01 -> precipitation_deviation_east_asia_period_1.png
  Period 2: 2026-06-02 to 2026-06-06 -> precipitation_deviation_east_asia_period_2.png
  Period 3: 2026-06-07 to 2026-06-11 -> precipitation_deviation_east_asia_period_3.png

Precipitation NaN coverage after interpolation: 0.00%


# run gatun lake code

In [33]:
import os
import shutil
import subprocess

# Cell 1: Clean up and preprocess Gatun Lake data

# Define paths
gatun_graphs_dir = r"P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_graphs"
gatun_data_dir = r"P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_lake_data"
gatun_script = r"P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_lake_main.py"

# Function to safely remove files from a directory
def clean_directory(directory):
    if os.path.exists(directory):
        try:
            # Remove all files in directory
            for filename in os.listdir(directory):
                file_path = os.path.join(directory, filename)
                try:
                    if os.path.isfile(file_path):
                        os.unlink(file_path)
                    elif os.path.isdir(file_path):
                        shutil.rmtree(file_path)
                    print(f"Removed: {file_path}")
                except Exception as e:
                    print(f"Error removing {file_path}: {e}")
        except Exception as e:
            print(f"Error accessing directory {directory}: {e}")
    else:
        print(f"Directory does not exist: {directory}")

# Clean up directories
print("Cleaning up Gatun Lake directories...")
clean_directory(gatun_graphs_dir)
clean_directory(gatun_data_dir)

# Run the Gatun Lake script
print("\nRunning Gatun Lake processing script...")
try:
    result = subprocess.run(['python', gatun_script], capture_output=True, text=True)
    if result.returncode == 0:
        print("Gatun Lake script completed successfully")
    else:
        print("Error running Gatun Lake script:")
        print(result.stderr)
except Exception as e:
    print(f"Error executing Gatun Lake script: {e}")

print("\nPreprocessing completed")



Cleaning up Gatun Lake directories...
Removed: P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_graphs\Gatun_Lake_Historical_Levels_2010_Onwards.png
Removed: P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_graphs\Gatun_Lake_Historical_Water_Levels.png
Removed: P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_graphs\Gatun_Lake_Obs_Fcst.png
Removed: P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_graphs\Gatun_Water_Level_Projections.png
Removed: P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_lake_data\Historical_Water_Levels_20260528.csv
Removed: P:\QFA\TonyWeather\Meteological_toolbox\All_year\gatun_lake\gatun_lake_data\Water_Level_Projection_20260528.csv

Running Gatun Lake processing script...
Gatun Lake script completed successfully

Preprocessing completed


# generate report

In [40]:
from fpdf import FPDF
from PIL import Image
import os
from PyPDF2 import PdfMerger
from pathlib import Path


def add_image(pdf, image_path, x, y, max_width, max_height):
    try:
        with Image.open(image_path) as img:
            img_width, img_height = img.size
    except Exception as e:
        print(f"Error opening {image_path}: {e}")
        return

    dpi = 96
    img_width_mm = img_width * 25.4 / dpi
    img_height_mm = img_height * 25.4 / dpi
    scale = min(max_width / img_width_mm, max_height / img_height_mm, 1)
    new_width = img_width_mm * scale
    new_height = img_height_mm * scale
    pdf.image(image_path, x=x, y=y, w=new_width, h=new_height)


print('creating PDF /7')
pdf = FPDF()
padding = 10

# -------------------------------
# PAGE 1: Maps for 3 periods per region
# -------------------------------
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "Temperature Outlook", ln=True, align="C")

maps_folder = r"P:\QFA\TonyWeather\marta\LPG_REPORT\maps"

periods = [1, 2, 3]

padding = 10
gap = 5
available_width = pdf.w - 2 * padding - (len(periods) - 1) * gap
each_width = available_width / len(periods)
current_y = pdf.get_y() + 5

map_rows = [
    ("east_asia", "Temperature Deviation", "temperature_deviation"),
    ("east_asia", "Precipitation Deviation", "precipitation_deviation"),
    ("us", "Temperature Deviation", "temperature_deviation"),
    ("europe", "Temperature Deviation", "temperature_deviation"),
]

for region, metric_label, file_prefix in map_rows:
    map_images = [
        os.path.join(maps_folder, f"{file_prefix}_{region}_period_{p}.png")
        for p in periods
    ]

    pdf.set_xy(padding, current_y - 4)
    pdf.set_font("Arial", size=10)
    pdf.cell(0, 5, f"{region.replace('_', ' ').title()} - {metric_label}")

    for i, img_path in enumerate(map_images):
        x = padding + i * (each_width + gap)
        add_image(pdf, img_path, x=x, y=current_y, max_width=each_width, max_height=50)

    current_y += 58

pdf.set_y(current_y)

# -------------------------------
# Three Gorges Dam catchment page
# -------------------------------
tg_map_png = r"P:\QFA\TonyWeather\marta\LPG_REPORT\maps\three_gorges_catchment_precip_map.png"
tg_line_png = r"P:\QFA\TonyWeather\marta\LPG_REPORT\maps\three_gorges_precip_deviation_line.png"
pdf.add_page()
pdf.set_font("Arial", "B", 16)
pdf.cell(0, 12, "Three Gorges Dam Catchment", ln=True, align="C")
pdf.set_font("Arial", size=10)
pdf.cell(0, 6, "Precipitation deviation from climatology over the upstream drainage basin", ln=True, align="C")
tg_y = pdf.get_y() + 4
tg_avail_w = pdf.w - 2 * padding
add_image(pdf, tg_map_png, x=padding, y=tg_y, max_width=tg_avail_w, max_height=120)
tg_y += 125
add_image(pdf, tg_line_png, x=padding, y=tg_y, max_width=tg_avail_w, max_height=85)

# --- Paired plots ---
# Left = CDD plot, right = Temperature plot.
pairs = [
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\Tallahassee_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\USA_Tallahassee_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\Columbia_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\USA_Columbia_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\Raleigh_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\USA_Raleigh_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\Kansas_and_Oklahoma_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\USA_Kansas,_Oklahoma_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\Japan_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\Japan_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\South_Korea_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\South_Korea_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\Germany_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\Germany_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\France_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\France_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\China_North_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\China_North_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\China_Central_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\China_Central_temperature.png"
    ),
    (
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\CDD_history_by_country\plots\China_South_summer_CDD_comparison.png",
        r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\China_South_temperature.png"
    )
]

pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "Paired Plots (continued)", ln=True, align="C")
current_y = pdf.get_y() + 5

pairs_page1 = pairs[:3]
col_width = (pdf.w - 2 * padding - 10) / 2
row_height = 60
for left_img, right_img in pairs_page1:
    add_image(pdf, left_img, x=padding, y=current_y, max_width=col_width, max_height=row_height)
    add_image(pdf, right_img, x=padding + col_width + 10, y=current_y, max_width=col_width, max_height=row_height)
    current_y += row_height + 10

print('page 2/8')
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "Paired Plots (continued) - Page 2", ln=True, align="C")
current_y = pdf.get_y() + 5
pairs_page2 = pairs[3:6]
for left_img, right_img in pairs_page2:
    add_image(pdf, left_img, x=padding, y=current_y, max_width=col_width, max_height=row_height)
    add_image(pdf, right_img, x=padding + col_width + 10, y=current_y, max_width=col_width, max_height=row_height)
    current_y += row_height + 10

print('page 3/8')
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "Paired Plots (continued) - Page 3", ln=True, align="C")
current_y = pdf.get_y() + 5
pairs_page3 = pairs[6:9]
for left_img, right_img in pairs_page3:
    add_image(pdf, left_img, x=padding, y=current_y, max_width=col_width, max_height=row_height)
    add_image(pdf, right_img, x=padding + col_width + 10, y=current_y, max_width=col_width, max_height=row_height)
    current_y += row_height + 10

print('page 4/10 - China CDD')
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "China Regions - CDD & Temperature", ln=True, align="C")
current_y = pdf.get_y() + 5
pairs_page4 = pairs[9:]
for left_img, right_img in pairs_page4:
    add_image(pdf, left_img, x=padding, y=current_y, max_width=col_width, max_height=row_height)
    add_image(pdf, right_img, x=padding + col_width + 10, y=current_y, max_width=col_width, max_height=row_height)
    current_y += row_height + 10

print('page 5/10')
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "Other Temperature Plots", ln=True, align="C")
unused_temp_plots = [
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\Brazil_Curitiba,_Santos_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\Brazil_Suape_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\China_Changde_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\China_Ganzhou_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\China_Guiyang_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\India_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\Morocco_temperature.png",
    r"P:\QFA\TonyWeather\marta\LPG_REPORT\temperature_plots\Panama_temperature.png"
]
cols = 2
gap = 10
col_width_unused = (pdf.w - 2 * padding - gap) / 2
row_height_unused = 60
current_y = pdf.get_y() + 5
for i, img_path in enumerate(unused_temp_plots):
    col = i % cols
    if i > 0 and col == 0:
        current_y += row_height_unused + 10
    x = padding + col * (col_width_unused + gap)
    add_image(pdf, img_path, x=x, y=current_y, max_width=col_width_unused, max_height=row_height_unused)

print('page 5/9')
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, "T2M Anomalies and Differences in seasonal forecast", ln=True, align="C")
plots_folder = r"P:\QFA\TonyWeather\marta\LPG_REPORT\PLOTS"
anomaly_plots = [
    os.path.join(plots_folder, f"T2M_anomaly_{month_labels[0]}.png"),
    os.path.join(plots_folder, f"T2M_anomaly_{month_labels[1]}.png"),
    os.path.join(plots_folder, f"T2M_anomaly_{month_labels[2]}.png"),
    os.path.join(plots_folder, f"T2M_anomaly_{month_labels[3]}.png"),
    os.path.join(plots_folder, f"T2M_anomaly_{month_labels[4]}.png")
]
diff_plots = [
    os.path.join(plots_folder, f"T2M_diff_from_mean_{month_labels[0]}.png"),
    os.path.join(plots_folder, f"T2M_diff_from_mean_{month_labels[1]}.png"),
    os.path.join(plots_folder, f"T2M_diff_from_mean_{month_labels[2]}.png"),
    os.path.join(plots_folder, f"T2M_diff_from_mean_{month_labels[3]}.png"),
    os.path.join(plots_folder, f"T2M_diff_from_mean_{month_labels[4]}.png")
]
col_width_plots = (pdf.w - 2 * padding - 10) / 2
row_height_plots = 60
current_y = pdf.get_y() + 5
for anomaly, diff in zip(anomaly_plots, diff_plots):
    add_image(pdf, anomaly, x=padding, y=current_y, max_width=col_width_plots, max_height=row_height_plots)
    add_image(pdf, diff, x=padding + col_width_plots + 10, y=current_y, max_width=col_width_plots, max_height=row_height_plots)
    current_y += row_height_plots + 10

print('page 6/9')
gatun_lake_plot = r"P:/QFA/TonyWeather/Meteological_toolbox/All_year/gatun_lake/gatun_graphs/Gatun_Lake_Obs_Fcst.png"
pdf.add_page()
pdf.set_font("Arial", "B", 16)
pdf.cell(210, 12, "Gatun Lake Levels", ln=True, align="C")
current_y = pdf.get_y() + 5
avail_width = pdf.w - 2 * padding
add_image(pdf, gatun_lake_plot, x=padding, y=current_y, max_width=avail_width, max_height=240)

reports_dir = Path(r"P:\QFA\TonyWeather\marta\LPG_REPORT\reports")
reports_dir.mkdir(parents=True, exist_ok=True)
date_str = datetime.today().strftime("%d_%m")
core_pdf = str(reports_dir / f"report_CDD_core_{date_str}.pdf")
pdf.output(core_pdf)

today = datetime.today().strftime("%Y-%m-%d")
external_padd_pdf = fr"P:\QFA\TonyWeather\marta\LPG_REPORT\PADD_plots\ridi_data\daily_CDD\CDD_combined\CDD_PADD_Report_{today}.pdf"
final_pdf = str(reports_dir / f"report_CDD_{date_str}.pdf")
fog_plots_pdf = "fog_section.pdf"

# fog_pdf = FPDF()

# filtered_images_dir = r"P:\QFA\TonyWeather\marta\PETROLEUM_REPORT\filtered_images"
# fog_images = sorted(
#     os.path.join(filtered_images_dir, f)
#     for f in os.listdir(filtered_images_dir)
#     if f.lower().endswith('.png')
# ) if os.path.exists(filtered_images_dir) else []

# fog_pdf.add_page()
# fog_pdf.set_font("Arial", "B", 16)
# fog_pdf.cell(0, 12, "Fog Conditions", ln=True, align="C")

# num_cols = 4
# fog_gap = 5
# fog_width = (fog_pdf.w - 2 * padding - (num_cols - 1) * fog_gap) / num_cols
# fog_height = 40
# current_y = fog_pdf.get_y() + 5
# col_ctr = 0

# for img in fog_images:
#     x = padding + col_ctr * (fog_width + fog_gap)
#     add_image(fog_pdf, img, x, current_y, fog_width, fog_height)
#     col_ctr += 1
#     if col_ctr == num_cols:
#         col_ctr = 0
#         current_y += fog_height + fog_gap

# fog_pdf.output(fog_plots_pdf)

merger = PdfMerger()
merger.append(core_pdf)
if os.path.exists(external_padd_pdf):
    merger.append(external_padd_pdf)
else:
    print(f"Skipping missing external PADD PDF: {external_padd_pdf}")
merger.append(fog_plots_pdf)

try:
    merger.write(final_pdf)
    output_pdf = final_pdf
except PermissionError:
    timestamp = datetime.today().strftime("%Y%m%d_%H%M%S")
    output_pdf = str(reports_dir / f"report_CDD_{date_str}_{timestamp}.pdf")
    merger.write(output_pdf)
    print(f"Default report locked; wrote fallback file: {output_pdf}")
finally:
    merger.close()

print("✅ Final PDF created:", output_pdf)

creating PDF /7
page 2/8
page 3/8
page 4/10 - China CDD
page 5/10
page 5/9
page 6/9
Skipping missing external PADD PDF: P:\QFA\TonyWeather\marta\LPG_REPORT\PADD_plots\ridi_data\daily_CDD\CDD_combined\CDD_PADD_Report_2026-05-28.pdf
✅ Final PDF created: report_CDD.pdf


In [35]:
# import os
# import tempfile
# import warnings
# import requests
# import geopandas as gpd
# from shapely.geometry import LineString
# import matplotlib.pyplot as plt

# warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# output_geojson = r"P:\QFA\TonyWeather\marta\LPG_REPORT\yangtze_basin_precise.geojson"
# output_plot = r"P:\QFA\TonyWeather\marta\LPG_REPORT\yangtze_basin_precise_preview.png"
# url = "https://data.hydrosheds.org/file/hydrobasins/standard/hybas_as_lev12_v1c.zip"
# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
# }

# print("Downloading HydroBASINS Asia level 12...")
# with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp:
#     tmp_zip = tmp.name

# response = requests.get(url, headers=headers, timeout=300, verify=False)
# response.raise_for_status()
# with open(tmp_zip, 'wb') as f:
#     f.write(response.content)

# print(f"Downloaded archive: {tmp_zip}")
# basins = gpd.read_file(tmp_zip)

# if basins.crs is None:
#     basins = basins.set_crs("EPSG:4326")
# elif basins.crs.to_string() != "EPSG:4326":
#     basins = basins.to_crs("EPSG:4326")

# if 'MAIN_BAS' in basins.columns:
#     yangtze = basins[basins['MAIN_BAS'].astype(str) == '3140405440']
# else:
#     yangtze = basins.iloc[0:0].copy()

# if yangtze.empty:
#     river_line = LineString([
#         (91.0, 34.5), (95.0, 33.8), (99.0, 32.0), (103.0, 30.8),
#         (107.0, 30.6), (111.0, 30.5), (114.5, 30.6), (118.5, 31.2), (121.7, 31.3)
#     ])
#     search_geom = gpd.GeoSeries([river_line], crs='EPSG:4326').buffer(0.6).iloc[0]
#     touched = basins[basins.intersects(search_geom)]

#     if touched.empty:
#         raise RuntimeError("Could not isolate Yangtze basin from HydroBASINS.")

#     if 'MAIN_BAS' in touched.columns:
#         main_bas = touched['MAIN_BAS'].astype(str).value_counts().index[0]
#         yangtze = basins[basins['MAIN_BAS'].astype(str) == main_bas]
#     else:
#         yangtze = touched.copy()

# yangtze_boundary = yangtze.dissolve(by=None)
# yangtze_boundary.to_file(output_geojson, driver='GeoJSON')

# fig, ax = plt.subplots(figsize=(10, 8))
# yangtze_boundary.boundary.plot(ax=ax, color='darkblue', linewidth=1.2)
# ax.set_title('Yangtze River Basin - HydroBASINS precise boundary')
# ax.set_xlabel('Longitude (°E)')
# ax.set_ylabel('Latitude (°N)')
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig(output_plot, dpi=150, bbox_inches='tight')
# plt.close(fig)

# print(f"Saved precise Yangtze boundary: {output_geojson}")
# print(f"Saved boundary preview: {output_plot}")
# print(f"Sub-basins dissolved: {len(yangtze)}")

# try:
#     os.remove(tmp_zip)
# except OSError:
#     pass

In [36]:
# import xarray as xr

# clim = xr.open_dataset(precip_clim_path)
# print('dims:', clim.dims)
# print('coords:', list(clim.coords))
# print('data_vars:', list(clim.data_vars))
# for var in clim.data_vars:
#     print(var, clim[var].dims, clim[var].shape)
# clim.close()